## 0. Task

13 rhetorical roles, macro-F1. **Selected:** efficient LoRMA + sentence BiLSTM-CRF + rarity bias.


In [ ]:
import os, sys, types, random, json, math, zipfile
from pathlib import Path

_BEFORE = set(globals()) | {"sys", "types", "_BEFORE", "_register"}

def _register(name):
    global _BEFORE
    mod = types.ModuleType(name)
    mod.__name__ = name
    mod.__file__ = name + ".py"
    for key, value in list(globals().items()):
        if key not in _BEFORE:
            setattr(mod, key, value)
    sys.modules[name] = mod
    _BEFORE = set(globals()) | {"sys", "types", "_BEFORE", "_register"}
    print("in this notebook:", name)

print("This notebook is self-contained (starter style). RUN_TRAINING = False for a viva.")


## 1. Check GPU


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))


## 2. Phase-1 body + LoRMA from Phase 2


In [ ]:
"""Byte-level BPE for CS781 Capstone Phase 1.

This is trained from scratch on the capstone corpus. It does not use
SentencePiece, Hugging Face tokenizers, or the GPT-2 vocabulary.

Assignment 1 merges the pair that occurs most often. On legal text that
fills the vocabulary with pieces of very common words such as "the" and
"of", even when those letters usually belong to other words.

Here a pair must occur at least `min_pair_count` times. Among those pairs
the one with the highest Dice-weighted count is merged:

    score(x, y) = C(xy) * 2 * C(xy) / (C(x) + C(y))

The fraction is near 1 when x and y almost always appear as this pair, and
near 0 when they usually appear apart. Multiplying by C(xy) keeps frequent
pairs ahead of rare ones. If two pairs have the same score, the higher count
wins. If the counts are also equal, the lexicographically smaller pair wins,
which is the Assignment 1 tie-break.

Merges stay inside a whitespace-separated word. A word-end marker is part of
that word, so a frequent whole word can become one token. Spaces stay
spaces, and uppercase stays uppercase. The first four vocabulary ids are
<pad>, <unk>, <s>, </s>. Every byte 0-255 has an id, so ordinary text does
not need <unk>.
"""

from __future__ import annotations

import heapq
import json
import re
from collections import Counter


END = 256  # word-end marker, not a UTF-8 byte
PAD_ID = 0
UNK_ID = 1
BOS_ID = 2
EOS_ID = 3
SPECIALS = ("<pad>", "<unk>", "<s>", "</s>")
# Model ids 0-3 are the specials above. Internal symbol 0 (byte 0) is model id 4.
ID_OFFSET = 4
BASE_SYMBOLS = 257  # bytes 0-255 plus END
TOKEN_RE = re.compile(r"\s+|\S+")


def iter_texts(records):
    """Yield strings from a list, a dict of records, or a single record."""
    if isinstance(records, list):
        rows = records
    elif isinstance(records, dict):
        if isinstance(records.get("text"), str):
            rows = [records]
        else:
            rows = list(records.values())
    else:
        raise TypeError("records must be a list or a dict")
    for row in rows:
        if isinstance(row, str):
            yield row
        elif isinstance(row, dict) and isinstance(row.get("text"), str):
            yield row["text"]


class LegalBPE:
    def __init__(self):
        self.score_name = "dice"
        self.min_pair_count = 2
        self.requested_vocab_size = ID_OFFSET + BASE_SYMBOLS
        self.merges = []  # list of (left_internal, right_internal), in learn order
        self.symbol_bytes = {i: (i,) for i in range(256)}
        self.symbol_bytes[END] = (END,)
        self.merge_rank = {}
        self.merge_to = {}
        self.keys = {}
        self.history = []
        self._cache = {}
        self._rebuild_tables()

    # ------------------------------------------------------------------
    # Public attributes the Phase 1 notebook uses
    # ------------------------------------------------------------------

    @property
    def pad_token_id(self):
        return PAD_ID

    @property
    def unk_token_id(self):
        return UNK_ID

    @property
    def bos_token_id(self):
        return BOS_ID

    @property
    def eos_token_id(self):
        return EOS_ID

    pad_token = SPECIALS[0]
    unk_token = SPECIALS[1]
    bos_token = SPECIALS[2]
    eos_token = SPECIALS[3]

    def __len__(self):
        return ID_OFFSET + BASE_SYMBOLS + len(self.merges)

    # ------------------------------------------------------------------
    # Training
    # ------------------------------------------------------------------

    @classmethod
    def train(cls, texts, vocab_size=8192, min_pair_count=2, score="dice", max_chars=None):
        if score not in ("dice", "frequency"):
            raise ValueError("score must be 'dice' or 'frequency'")
        minimum = ID_OFFSET + BASE_SYMBOLS + 1
        if vocab_size < minimum:
            raise ValueError(f"vocab_size must be at least {minimum}")

        model = cls()
        model.score_name = score
        model.min_pair_count = int(min_pair_count)
        model.requested_vocab_size = int(vocab_size)
        target_merges = int(vocab_size) - (ID_OFFSET + BASE_SYMBOLS)

        word_freq = Counter()
        documents = 0
        seen_chars = 0
        for text in texts:
            documents += 1
            if not isinstance(text, str) or text == "":
                continue
            if max_chars is not None and seen_chars >= max_chars:
                break
            seen_chars += len(text)
            for tok in TOKEN_RE.findall(text):
                word_freq[model._token_symbols(tok)] += 1
            if documents % 100000 == 0:
                print(f"BPE counted {documents:,} documents, {len(word_freq):,} unique words")

        if not word_freq:
            raise ValueError("BPE training saw no text")

        print(
            f"BPE documents: {documents:,} | characters: {seen_chars:,} | "
            f"unique words: {len(word_freq):,} | score: {score} | target vocab: {vocab_size}"
        )
        model._learn(word_freq, target_merges)
        print(f"BPE finished. Vocabulary size: {len(model):,}. Merges: {len(model.merges):,}.")
        for left, right, count, value in model.history[:12]:
            print(f"  merge {left!r} + {right!r}  count={count}  score={value:.2f}")
        return model

    def _learn(self, word_freq, target_merges):
        self.word_freq = dict(word_freq)
        self.pair_counts = Counter()
        self.pair_words = {}
        self.sym_counts = Counter()
        self.sym_pairs = {}
        self.version = {}
        self.heap = []
        self.history = []
        self.merges = []
        self.merge_rank = {}
        self.merge_to = {}
        self._cache = {}
        self.symbol_bytes = {i: (i,) for i in range(256)}
        self.symbol_bytes[END] = (END,)
        self.keys = {i: self._key_for(i) for i in range(257)}

        dirty_pairs = set()
        dirty_syms = set()
        for word, freq in self.word_freq.items():
            self._register(word, freq, dirty_pairs, dirty_syms)
        self._rebuild_heap()

        next_id = 257
        while len(self.merges) < target_merges:
            if len(self.heap) > max(1000, 8 * len(self.pair_counts)):
                self._rebuild_heap()
            chosen = self._pop_best()
            if chosen is None:
                print(
                    f"BPE stopped early at vocab {len(self)}. "
                    f"No pair occurs at least {self.min_pair_count} times."
                )
                break
            left, right = chosen
            count = self.pair_counts[chosen]
            value = self._score(chosen)
            new_id = next_id
            next_id += 1
            self.symbol_bytes[new_id] = self.symbol_bytes[left] + self.symbol_bytes[right]
            self.keys[new_id] = self._key_for(new_id)
            self.merges.append((left, right))
            self.history.append((self._pretty(left), self._pretty(right), int(count), float(value)))

            dirty_pairs = set()
            dirty_syms = set()
            for word in list(self.pair_words.get(chosen, ())):
                freq = self.word_freq.pop(word, None)
                if freq is None:
                    continue
                self._unregister(word, freq, dirty_pairs, dirty_syms)
                merged = _merge_symbols(word, left, right, new_id)
                if merged in self.word_freq:
                    self.word_freq[merged] += freq
                    self._register(merged, freq, dirty_pairs, dirty_syms)
                else:
                    self.word_freq[merged] = freq
                    self._register(merged, freq, dirty_pairs, dirty_syms)

            affected = set(dirty_pairs)
            for sym in dirty_syms:
                affected.update(self.sym_pairs.get(sym, ()))
            affected.add(chosen)
            for pair in affected:
                self._push(pair)

            done = len(self.merges)
            if done <= 10 or done % 500 == 0:
                print(
                    f"  [{done}/{target_merges}] {self.history[-1][0]!r} + "
                    f"{self.history[-1][1]!r}  count={count}  score={value:.2f}"
                )

        self._rebuild_tables()
        # Training indexes are large. Drop them before save.
        self.word_freq = {}
        self.pair_counts = Counter()
        self.pair_words = {}
        self.sym_counts = Counter()
        self.sym_pairs = {}
        self.heap = []

    def _register(self, word, freq, dirty_pairs, dirty_syms):
        for sym in word:
            self.sym_counts[sym] += freq
            dirty_syms.add(sym)
        for pair, times in Counter(zip(word, word[1:])).items():
            self.pair_counts[pair] += freq * times
            self.pair_words.setdefault(pair, set()).add(word)
            for sym in pair:
                self.sym_pairs.setdefault(sym, set()).add(pair)
            dirty_pairs.add(pair)

    def _unregister(self, word, freq, dirty_pairs, dirty_syms):
        for sym in word:
            self.sym_counts[sym] -= freq
            if self.sym_counts[sym] <= 0:
                del self.sym_counts[sym]
            dirty_syms.add(sym)
        for pair, times in Counter(zip(word, word[1:])).items():
            self.pair_counts[pair] -= freq * times
            words = self.pair_words.get(pair)
            if words is not None:
                words.discard(word)
                if not words:
                    del self.pair_words[pair]
            if self.pair_counts[pair] <= 0:
                del self.pair_counts[pair]
                for sym in pair:
                    bucket = self.sym_pairs.get(sym)
                    if bucket is not None:
                        bucket.discard(pair)
                        if not bucket:
                            del self.sym_pairs[sym]
                self.version[pair] = self.version.get(pair, 0) + 1
            else:
                dirty_pairs.add(pair)

    def _score(self, pair):
        count = self.pair_counts.get(pair, 0)
        if count < self.min_pair_count:
            return None
        if self.score_name == "frequency":
            return float(count)
        left = self.sym_counts.get(pair[0], 0)
        right = self.sym_counts.get(pair[1], 0)
        if left <= 0 or right <= 0:
            return None
        return (2.0 * count * count) / (left + right)

    def _push(self, pair):
        value = self._score(pair)
        if value is None:
            return
        version = self.version.get(pair, 0) + 1
        self.version[pair] = version
        count = self.pair_counts[pair]
        heapq.heappush(
            self.heap,
            (
                -value,
                -count,
                self.keys[pair[0]],
                self.keys[pair[1]],
                pair[0],
                pair[1],
                version,
            ),
        )

    def _rebuild_heap(self):
        self.heap = []
        for pair in list(self.pair_counts):
            self._push(pair)

    def _pop_best(self):
        while self.heap:
            _neg_value, _neg_count, _left_key, _right_key, left, right, version = heapq.heappop(self.heap)
            pair = (left, right)
            if self.version.get(pair) != version:
                continue
            if self._score(pair) is None:
                continue
            return pair
        return None

    def _rebuild_tables(self):
        self.merge_rank = {}
        self.merge_to = {}
        self.symbol_bytes = {i: (i,) for i in range(256)}
        self.symbol_bytes[END] = (END,)
        next_id = 257
        for rank, (left, right) in enumerate(self.merges):
            self.merge_rank[(left, right)] = rank
            self.merge_to[(left, right)] = next_id
            self.symbol_bytes[next_id] = self.symbol_bytes[left] + self.symbol_bytes[right]
            next_id += 1
        self.keys = {i: self._key_for(i) for i in self.symbol_bytes}
        self._cache = {}

    def _key_for(self, sym):
        parts = []
        for piece in self.symbol_bytes[sym]:
            parts.append("</w>" if piece == END else f"{piece:02x}")
        return "".join(parts)

    def _pretty(self, sym):
        raw = []
        end = False
        for piece in self.symbol_bytes[sym]:
            if piece == END:
                end = True
            else:
                raw.append(piece)
        try:
            text = bytes(raw).decode("utf-8")
            shown = text.encode("ascii", errors="backslashreplace").decode("ascii")
        except UnicodeDecodeError:
            shown = "".join(f"<{byte:02x}>" for byte in raw)
        return shown + "</w>" if end else shown

    def _token_symbols(self, token):
        raw = tuple(token.encode("utf-8"))
        if token.isspace():
            return raw
        return raw + (END,)

    # ------------------------------------------------------------------
    # Encode / decode
    # ------------------------------------------------------------------

    def encode(self, text, add_special_tokens=False):
        ids = []
        if add_special_tokens:
            ids.append(BOS_ID)
        if text:
            for tok in TOKEN_RE.findall(text):
                ids.extend(self._encode_token(tok))
        if add_special_tokens:
            ids.append(EOS_ID)
        return ids

    def encode_with_offsets(self, text):
        """Map text to token ids and the exact character span of each token.

        Merges never cross a whitespace boundary, and the word-end marker
        does not consume a character. A span is a half-open [start, end)
        into `text`, which is the same convention as the Phase 2 gold files.
        """
        ids = []
        spans = []
        for match in TOKEN_RE.finditer(text):
            token = match.group()
            origin = match.start()
            char_of_byte = []
            for index, char in enumerate(token):
                char_of_byte.extend([origin + index] * len(char.encode("utf-8")))
            byte_at = 0
            for model_id in self._encode_token(token):
                pieces = self.symbol_bytes.get(int(model_id) - ID_OFFSET, ())
                nbytes = sum(1 for piece in pieces if piece != END)
                if nbytes <= 0 or byte_at >= len(char_of_byte):
                    continue
                stop = min(byte_at + nbytes, len(char_of_byte))
                left = char_of_byte[byte_at]
                right = char_of_byte[stop - 1] + 1
                byte_at = stop
                if right > left:
                    ids.append(int(model_id))
                    spans.append((left, right))
        return ids, spans

    def _encode_token(self, token):
        cached = self._cache.get(token)
        if cached is None:
            symbols = list(self._token_symbols(token))
            while len(symbols) >= 2:
                best_rank = None
                best_index = None
                for i in range(len(symbols) - 1):
                    rank = self.merge_rank.get((symbols[i], symbols[i + 1]))
                    if rank is not None and (best_rank is None or rank < best_rank):
                        best_rank = rank
                        best_index = i
                if best_index is None:
                    break
                pair = (symbols[best_index], symbols[best_index + 1])
                symbols = symbols[:best_index] + [self.merge_to[pair]] + symbols[best_index + 2 :]
            cached = tuple(sym + ID_OFFSET for sym in symbols)
            self._cache[token] = cached
        return cached

    def decode(self, ids, skip_special_tokens=True):
        raw = []
        for model_id in ids:
            model_id = int(model_id)
            if model_id < ID_OFFSET:
                continue
            internal = model_id - ID_OFFSET
            pieces = self.symbol_bytes.get(internal)
            if pieces is None:
                continue
            for piece in pieces:
                if piece != END:
                    raw.append(piece)
        return bytes(raw).decode("utf-8", errors="replace")

    def batch_encode(self, texts, max_length=None, padding="left"):
        """Pad a batch for generation. Keep the end of a long prompt."""
        rows = []
        for text in texts:
            ids = self.encode(text, add_special_tokens=False)
            if max_length is not None and len(ids) > max_length:
                ids = ids[-max_length:]
            rows.append(ids)
        width = max((len(row) for row in rows), default=0)
        input_ids = []
        attention_mask = []
        for row in rows:
            pad = width - len(row)
            if padding == "left":
                input_ids.append([PAD_ID] * pad + row)
                attention_mask.append([0] * pad + [1] * len(row))
            else:
                input_ids.append(row + [PAD_ID] * pad)
                attention_mask.append([1] * len(row) + [0] * pad)
        return input_ids, attention_mask

    # ------------------------------------------------------------------
    # Save / load
    # ------------------------------------------------------------------

    def save(self, path):
        payload = {
            "format": "cs781-legal-bpe-v1",
            "score": self.score_name,
            "min_pair_count": self.min_pair_count,
            "vocab_size": len(self),
            "specials": list(SPECIALS),
            "merges": self.merges,
            "preview": [f"{left} + {right}" for left, right, _count, _value in self.history[:40]],
        }
        with open(path, "w", encoding="utf-8") as handle:
            json.dump(payload, handle, ensure_ascii=False)
        print("Saved BPE to", path)

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as handle:
            payload = json.load(handle)
        if payload.get("format") != "cs781-legal-bpe-v1":
            raise ValueError(f"{path} is not a cs781-legal-bpe-v1 file")
        model = cls()
        model.score_name = payload.get("score", "dice")
        model.min_pair_count = int(payload.get("min_pair_count", 2))
        model.merges = [(int(left), int(right)) for left, right in payload["merges"]]
        model.requested_vocab_size = int(payload.get("vocab_size", len(model)))
        model.history = []
        model._rebuild_tables()
        return model


def _merge_symbols(word, left, right, new_id):
    merged = []
    index = 0
    size = len(word)
    while index < size:
        if index + 1 < size and word[index] == left and word[index + 1] == right:
            merged.append(new_id)
            index += 2
        else:
            merged.append(word[index])
            index += 1
    return tuple(merged)


def _check():
    sticky = ["ab"] * 6
    loose = ["mn", "mp", "mq", "mr", "ms"] * 10
    one_merge = ID_OFFSET + BASE_SYMBOLS + 1

    dice = LegalBPE.train(sticky + loose, vocab_size=one_merge, min_pair_count=2, score="dice")
    freq = LegalBPE.train(sticky + loose, vocab_size=one_merge, min_pair_count=2, score="frequency")
    assert dice.history[0][0] == "a" and dice.history[0][1] == "b", dice.history[0]
    assert freq.history[0][0] in list("mnmpqrms") or freq.history[0][0].startswith("m"), freq.history[0]
    assert not (freq.history[0][0] == "a" and freq.history[0][1] == "b"), freq.history[0]

    samples = [
        "The Supreme Court held that the petitioner is entitled to relief.",
        "Section 302 of the Indian Penal Code\nwas applied.",
        "court Court COURT",
        "न्यायालय ने आदेश दिया।",
        "  spaced   words\tand tabs",
    ]
    full = LegalBPE.train(samples * 30, vocab_size=400, min_pair_count=2, score="dice")
    for text in samples:
        assert full.decode(full.encode(text)) == text, text
    assert full.encode("Court") != full.encode("court")
    joined = full.decode(full.encode("ab cd"))
    assert joined == "ab cd"
    # A space is its own token, so a merge cannot glue the b in "ab" to the c in "cd".
    encoded = full.encode("ab cd")
    assert len(encoded) >= 2

    path = "_legal_bpe_selfcheck.json"
    full.save(path)
    loaded = LegalBPE.load(path)
    for text in samples:
        assert loaded.decode(loaded.encode(text)) == text
    assert len(loaded) == len(full)
    ids, mask = loaded.batch_encode(["short", "a longer prompt"], max_length=8, padding="left")
    assert len(ids[0]) == len(ids[1])
    assert mask[0][0] == 0 or len(loaded.encode("short")) == len(ids[0])
    import os
    os.remove(path)
    print("legal_bpe self-check passed")


In [ ]:
_register('legal_bpe')


In [ ]:
"""Four Phase-1 language models, all trained from scratch.

They share a token embedding with the output layer. None of them loads a
pretrained checkpoint, and none of them uses nn.Transformer or
nn.MultiheadAttention.

The four designs are different in structure, not just in width or learning rate:

decoder
    Causal Transformer. A token may look only at earlier tokens. Position is a
    learned vector. The feed-forward layer is a GELU MLP. Pre-norm LayerNorm.

encdec
    The first half of a training block is read by a bidirectional encoder.
    A causal decoder writes the second half and cross-attends to the encoder.
    At test time the prompt is encoded, then the continuation is decoded.

hybrid
    Same size as the decoder, but most layers see only a local window of
    earlier tokens. Every fourth layer is full causal attention, so a long
    legal reference can still move through the network.

efficient
    RoPE instead of learned positions, RMSNorm instead of LayerNorm, SwiGLU
    instead of GELU, and grouped-query attention: 16 query heads share 4
    key/value heads.
"""

from __future__ import annotations

import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint


VOCAB_SIZE = 8192
BLOCK = 1024


def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters())


class RMSNorm(nn.Module):
    def __init__(self, n_embd, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(n_embd))
        self.eps = eps

    def forward(self, x):
        scale = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return x * scale * self.weight


def _init_linear(module):
    if isinstance(module, nn.Linear):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if module.bias is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)


def apply_rope(x):
    # x: batch, heads, time, dim. Pairs (0,1), (2,3), ... are rotated by position.
    _batch, _heads, steps, dim = x.shape
    half = dim // 2
    freq = 1.0 / (10000 ** (torch.arange(half, device=x.device, dtype=torch.float32) / half))
    pos = torch.arange(steps, device=x.device, dtype=torch.float32)
    angles = torch.outer(pos, freq)
    cos = angles.cos()[None, None, :, :]
    sin = angles.sin()[None, None, :, :]
    even = x[..., 0::2].float()
    odd = x[..., 1::2].float()
    rotated_even = even * cos - odd * sin
    rotated_odd = even * sin + odd * cos
    stacked = torch.stack((rotated_even, rotated_odd), dim=-1)
    return stacked.flatten(-2).to(dtype=x.dtype)


class CausalAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, window=None):
        super().__init__()
        if n_embd % n_head != 0:
            raise ValueError("n_embd must divide into equal heads")
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.window = window
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.bidirectional = False
        causal = torch.triu(torch.ones(block_size, block_size, dtype=torch.bool), diagonal=1)
        self.register_buffer("causal", causal, persistent=False)

    def forward(self, x, pad_mask=None):
        batch, steps, channels = x.shape
        qkv = self.qkv(x).view(batch, steps, 3, self.n_head, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        query, key, value = qkv[0], qkv[1], qkv[2]
        scores = (query @ key.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if not self.bidirectional:
            banned = self.causal[:steps, :steps]
            if self.window is not None:
                positions = torch.arange(steps, device=x.device)
                too_far = positions[None, :] < (positions[:, None] - (self.window - 1))
                banned = banned | too_far
            scores = scores.masked_fill(banned, float("-inf"))
        if pad_mask is not None:
            scores = scores.masked_fill(~pad_mask[:, None, None, :], float("-inf"))
        weights = torch.nan_to_num(torch.softmax(scores, dim=-1), nan=0.0)
        mixed = (weights @ value).transpose(1, 2).contiguous().view(batch, steps, channels)
        return self.proj(mixed)


class GroupedQueryAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, block_size):
        super().__init__()
        if n_embd % n_head != 0 or n_head % n_kv_head != 0:
            raise ValueError("heads must divide the width and the kv heads")
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.head_dim = n_embd // n_head
        self.q = nn.Linear(n_embd, n_embd, bias=False)
        self.k = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.v = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.bidirectional = False
        causal = torch.triu(torch.ones(block_size, block_size, dtype=torch.bool), diagonal=1)
        self.register_buffer("causal", causal, persistent=False)

    def forward(self, x, pad_mask=None):
        batch, steps, _channels = x.shape
        query = self.q(x).view(batch, steps, self.n_head, self.head_dim).transpose(1, 2)
        key = self.k(x).view(batch, steps, self.n_kv_head, self.head_dim).transpose(1, 2)
        value = self.v(x).view(batch, steps, self.n_kv_head, self.head_dim).transpose(1, 2)
        query = apply_rope(query)
        key = apply_rope(key)
        groups = self.n_head // self.n_kv_head
        key = key.repeat_interleave(groups, dim=1)
        value = value.repeat_interleave(groups, dim=1)
        scores = (query @ key.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if not self.bidirectional:
            scores = scores.masked_fill(self.causal[:steps, :steps], float("-inf"))
        if pad_mask is not None:
            scores = scores.masked_fill(~pad_mask[:, None, None, :], float("-inf"))
        weights = torch.nan_to_num(torch.softmax(scores, dim=-1), nan=0.0)
        mixed = (weights @ value).transpose(1, 2).contiguous().view(batch, steps, -1)
        return self.proj(mixed)


class CrossAttention(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.q = nn.Linear(n_embd, n_embd, bias=False)
        self.k = nn.Linear(n_embd, n_embd, bias=False)
        self.v = nn.Linear(n_embd, n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)

    def forward(self, x, memory, memory_mask=None):
        batch, steps, channels = x.shape
        mem_steps = memory.size(1)
        query = self.q(x).view(batch, steps, self.n_head, self.head_dim).transpose(1, 2)
        key = self.k(memory).view(batch, mem_steps, self.n_head, self.head_dim).transpose(1, 2)
        value = self.v(memory).view(batch, mem_steps, self.n_head, self.head_dim).transpose(1, 2)
        scores = (query @ key.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if memory_mask is not None:
            scores = scores.masked_fill(~memory_mask[:, None, None, :], float("-inf"))
        weights = torch.nan_to_num(torch.softmax(scores, dim=-1), nan=0.0)
        mixed = (weights @ value).transpose(1, 2).contiguous().view(batch, steps, channels)
        return self.proj(mixed)


class BidirectionalAttention(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)

    def forward(self, x, pad_mask=None):
        batch, steps, channels = x.shape
        qkv = self.qkv(x).view(batch, steps, 3, self.n_head, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        query, key, value = qkv[0], qkv[1], qkv[2]
        scores = (query @ key.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if pad_mask is not None:
            scores = scores.masked_fill(~pad_mask[:, None, None, :], float("-inf"))
        weights = torch.nan_to_num(torch.softmax(scores, dim=-1), nan=0.0)
        mixed = (weights @ value).transpose(1, 2).contiguous().view(batch, steps, channels)
        return self.proj(mixed)


class GeluMLP(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.up = nn.Linear(n_embd, 4 * n_embd)
        self.down = nn.Linear(4 * n_embd, n_embd)

    def forward(self, x):
        return self.down(F.gelu(self.up(x)))


class SwiGLU(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        hidden = int(8 * n_embd / 3)
        self.gate = nn.Linear(n_embd, hidden, bias=False)
        self.up = nn.Linear(n_embd, hidden, bias=False)
        self.down = nn.Linear(hidden, n_embd, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


class DecoderBlock(nn.Module):
    def __init__(self, n_embd, n_head, block_size, window=None, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(n_embd)
        self.attn = CausalAttention(n_embd, n_head, block_size, window=window)
        self.norm2 = nn.LayerNorm(n_embd)
        self.mlp = GeluMLP(n_embd)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, pad_mask=None):
        x = x + self.drop(self.attn(self.norm1(x), pad_mask=pad_mask))
        x = x + self.drop(self.mlp(self.norm2(x)))
        return x


class EfficientBlock(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, block_size, dropout=0.1):
        super().__init__()
        self.norm1 = RMSNorm(n_embd)
        self.attn = GroupedQueryAttention(n_embd, n_head, n_kv_head, block_size)
        self.norm2 = RMSNorm(n_embd)
        self.mlp = SwiGLU(n_embd)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, pad_mask=None):
        x = x + self.drop(self.attn(self.norm1(x), pad_mask=pad_mask))
        x = x + self.drop(self.mlp(self.norm2(x)))
        return x


class EncoderBlock(nn.Module):
    def __init__(self, n_embd, n_head, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(n_embd)
        self.attn = BidirectionalAttention(n_embd, n_head)
        self.norm2 = nn.LayerNorm(n_embd)
        self.mlp = GeluMLP(n_embd)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, pad_mask=None):
        x = x + self.drop(self.attn(self.norm1(x), pad_mask=pad_mask))
        x = x + self.drop(self.mlp(self.norm2(x)))
        return x


class DecoderCrossBlock(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(n_embd)
        self.self_attn = CausalAttention(n_embd, n_head, block_size)
        self.norm2 = nn.LayerNorm(n_embd)
        self.cross_attn = CrossAttention(n_embd, n_head)
        self.norm3 = nn.LayerNorm(n_embd)
        self.mlp = GeluMLP(n_embd)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, memory, pad_mask=None, memory_mask=None):
        x = x + self.drop(self.self_attn(self.norm1(x), pad_mask=pad_mask))
        x = x + self.drop(self.cross_attn(self.norm2(x), memory, memory_mask=memory_mask))
        x = x + self.drop(self.mlp(self.norm3(x)))
        return x


class TiedLM(nn.Module):
    def __init__(self, vocab_size, n_embd, block_size, use_positions=True):
        super().__init__()
        self.block_size = block_size
        self.embed = nn.Embedding(vocab_size, n_embd)
        self.pos = nn.Embedding(block_size, n_embd) if use_positions else None
        self.drop = nn.Dropout(0.1)
        self.norm = None

    def token_plus_position(self, idx):
        x = self.embed(idx)
        if self.pos is not None:
            steps = idx.size(1)
            positions = torch.arange(steps, device=idx.device)
            x = x + self.pos(positions)
        return self.drop(x)

    def logits_from(self, x):
        x = self.norm(x)
        return F.linear(x, self.embed.weight)

    def run_blocks(self, blocks, x, pad_mask=None):
        for block in blocks:
            if self.training:
                x = checkpoint(block, x, pad_mask, use_reentrant=False)
            else:
                x = block(x, pad_mask=pad_mask)
        return x


class DecoderLM(TiedLM):
    def __init__(self, vocab_size=VOCAB_SIZE, n_embd=1024, n_layer=8, n_head=16, block_size=BLOCK, windowed=False):
        super().__init__(vocab_size, n_embd, block_size, use_positions=True)
        blocks = []
        for layer in range(n_layer):
            window = None
            if windowed and layer % 4 != 3:
                window = 256
            blocks.append(DecoderBlock(n_embd, n_head, block_size, window=window))
        self.blocks = nn.ModuleList(blocks)
        self.norm = nn.LayerNorm(n_embd)
        self.apply(_init_linear)

    def forward(self, idx, labels=None, pad_mask=None):
        if idx.size(1) > self.block_size:
            raise ValueError("sequence is longer than the context")
        x = self.run_blocks(self.blocks, self.token_plus_position(idx), pad_mask)
        logits = self.logits_from(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits[:, :-1].reshape(-1, logits.size(-1)), labels[:, 1:].reshape(-1))
        return logits, loss


class EncoderDecoderLM(TiedLM):
    def __init__(self, vocab_size=VOCAB_SIZE, n_embd=768, n_layer=6, n_head=12, block_size=BLOCK):
        super().__init__(vocab_size, n_embd, block_size, use_positions=True)
        self.encoder = nn.ModuleList(EncoderBlock(n_embd, n_head) for _ in range(n_layer))
        self.decoder = nn.ModuleList(DecoderCrossBlock(n_embd, n_head, block_size) for _ in range(n_layer))
        self.norm = nn.LayerNorm(n_embd)
        self.enc_norm = nn.LayerNorm(n_embd)
        self.apply(_init_linear)

    def encode(self, idx, pad_mask=None):
        x = self.token_plus_position(idx)
        x = self.run_blocks(self.encoder, x, pad_mask)
        return self.enc_norm(x)

    def decode(self, idx, memory, pad_mask=None, memory_mask=None):
        x = self.token_plus_position(idx)
        for block in self.decoder:
            if self.training:
                x = checkpoint(block, x, memory, pad_mask, memory_mask, use_reentrant=False)
            else:
                x = block(x, memory, pad_mask=pad_mask, memory_mask=memory_mask)
        return self.logits_from(x)

    def forward(self, idx, labels=None, pad_mask=None):
        steps = idx.size(1)
        if steps < 2 or steps > self.block_size:
            raise ValueError("encoder-decoder training needs a block of length 2..context")
        split = steps // 2
        memory = self.encode(idx[:, :split], None if pad_mask is None else pad_mask[:, :split])
        dec_in = idx[:, split - 1 : -1]
        target = idx[:, split:]
        logits = self.decode(dec_in, memory)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), target.reshape(-1))
        return logits, loss


class EfficientLM(TiedLM):
    def __init__(self, vocab_size=VOCAB_SIZE, n_embd=1024, n_layer=10, n_head=16, n_kv_head=4, block_size=BLOCK):
        super().__init__(vocab_size, n_embd, block_size, use_positions=False)
        self.blocks = nn.ModuleList(
            EfficientBlock(n_embd, n_head, n_kv_head, block_size) for _ in range(n_layer)
        )
        self.norm = RMSNorm(n_embd)
        self.apply(_init_linear)

    def forward(self, idx, labels=None, pad_mask=None):
        if idx.size(1) > self.block_size:
            raise ValueError("sequence is longer than the context")
        x = self.run_blocks(self.blocks, self.token_plus_position(idx), pad_mask)
        logits = self.logits_from(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits[:, :-1].reshape(-1, logits.size(-1)), labels[:, 1:].reshape(-1))
        return logits, loss


def build_model(name, vocab_size=VOCAB_SIZE):
    if name == "decoder":
        model = DecoderLM(vocab_size=vocab_size)
    elif name == "encdec":
        model = EncoderDecoderLM(vocab_size=vocab_size)
    elif name == "hybrid":
        model = DecoderLM(vocab_size=vocab_size, windowed=True)
    elif name == "efficient":
        model = EfficientLM(vocab_size=vocab_size)
    else:
        raise ValueError("architecture must be decoder, encdec, hybrid, or efficient")
    total = count_parameters(model)
    if not 50_000_000 <= total <= 300_000_000:
        raise RuntimeError(f"{name} has {total} parameters, outside 50M-300M")
    return model


def sample_top_p(logits, top_p):
    sorted_logits, sorted_index = torch.sort(logits, descending=True, dim=-1)
    probs = torch.softmax(sorted_logits, dim=-1)
    cumulative = torch.cumsum(probs, dim=-1)
    remove = cumulative > top_p
    remove[..., 1:] = remove[..., :-1].clone()
    remove[..., 0] = False
    sorted_logits = sorted_logits.masked_fill(remove, float("-inf"))
    choice = torch.multinomial(torch.softmax(sorted_logits, dim=-1), num_samples=1)
    return torch.gather(sorted_index, -1, choice).squeeze(-1)


def _apply_stop(stop_fn, produced, finished, step, check_every):
    if stop_fn is None or (step + 1) % check_every != 0:
        return finished
    for position in torch.where(~finished)[0].tolist():
        if stop_fn(produced[position]):
            finished[position] = True
    return finished


@torch.no_grad()
def generate_causal(model, idx, max_new_tokens, temperature, top_p, eos_id=None, pad_id=0, stop_fn=None, check_every=32, min_new_tokens=0):
    pad_mask = idx != pad_id
    lengths = pad_mask.long().sum(dim=1).clamp(min=1)
    finished = torch.zeros(idx.size(0), dtype=torch.bool, device=idx.device)
    produced = [[] for _ in range(idx.size(0))]
    if idx.size(1) > model.block_size:
        overflow = idx.size(1) - model.block_size
        idx = idx[:, overflow:]
        pad_mask = pad_mask[:, overflow:]
        lengths = (lengths - overflow).clamp(min=1, max=model.block_size)
    for step in range(max_new_tokens):
        active = ~finished
        if not bool(active.any()):
            break
        if bool((active & (lengths >= model.block_size)).any()):
            idx = idx[:, 1:]
            pad_mask = pad_mask[:, 1:]
            lengths = (lengths - 1).clamp(min=1)
        if bool((lengths >= idx.size(1)).any()):
            idx = torch.cat([idx, idx.new_full((idx.size(0), 1), pad_id)], dim=1)
            pad_mask = torch.cat(
                [pad_mask, torch.zeros(idx.size(0), 1, dtype=torch.bool, device=idx.device)],
                dim=1,
            )
        limit = int(lengths.max().item())
        logits, _ = model(idx[:, :limit], pad_mask=pad_mask[:, :limit])
        last = (lengths - 1).clamp(min=0)
        row = torch.arange(idx.size(0), device=idx.device)
        next_logits = logits[row, last].float()
        if min_new_tokens and eos_id is not None:
            for position in range(idx.size(0)):
                if len(produced[position]) < min_new_tokens:
                    next_logits[position, eos_id] = -1e9
        if temperature <= 0:
            next_id = torch.argmax(next_logits, dim=-1)
        else:
            next_id = sample_top_p(next_logits / temperature, top_p)
        idx[row[active], lengths[active]] = next_id[active]
        pad_mask[row[active], lengths[active]] = True
        for position in torch.where(active)[0].tolist():
            produced[position].append(int(next_id[position]))
        lengths = lengths + active.long()
        if eos_id is not None:
            finished = finished | (active & (next_id == eos_id))
        finished = _apply_stop(stop_fn, produced, finished, step, check_every)
    return produced


@torch.no_grad()
def generate_encdec(model, idx, max_new_tokens, temperature, top_p, eos_id=None, pad_id=0, stop_fn=None, check_every=32):
    pad_mask = idx != pad_id
    lengths = pad_mask.long().sum(dim=1).clamp(min=1)
    if idx.size(1) > model.block_size:
        overflow = idx.size(1) - model.block_size
        idx = idx[:, overflow:]
        pad_mask = pad_mask[:, overflow:]
        lengths = (lengths - overflow).clamp(min=1)
    memory = model.encode(idx, pad_mask)
    row = torch.arange(idx.size(0), device=idx.device)
    dec = idx[row, lengths - 1].unsqueeze(1)
    produced = [[] for _ in range(idx.size(0))]
    finished = torch.zeros(idx.size(0), dtype=torch.bool, device=idx.device)
    for step in range(max_new_tokens):
        active = ~finished
        if not bool(active.any()):
            break
        if dec.size(1) >= model.block_size:
            dec = dec[:, 1:]
        logits = model.decode(dec, memory, memory_mask=pad_mask)
        next_logits = logits[:, -1, :]
        if temperature <= 0:
            next_id = torch.argmax(next_logits, dim=-1)
        else:
            next_id = sample_top_p(next_logits / temperature, top_p)
        next_id = torch.where(active, next_id, torch.full_like(next_id, eos_id or pad_id))
        dec = torch.cat([dec, next_id.unsqueeze(1)], dim=1)
        for position in torch.where(active)[0].tolist():
            produced[position].append(int(next_id[position]))
        if eos_id is not None:
            finished = finished | (active & (next_id == eos_id))
        finished = _apply_stop(stop_fn, produced, finished, step, check_every)
    return produced


def generate(model, idx, max_new_tokens, temperature, top_p, eos_id=None, pad_id=0, stop_fn=None, min_new_tokens=0):
    if isinstance(model, EncoderDecoderLM):
        return generate_encdec(model, idx, max_new_tokens, temperature, top_p, eos_id, pad_id, stop_fn)
    return generate_causal(model, idx, max_new_tokens, temperature, top_p, eos_id, pad_id, stop_fn,
                           min_new_tokens=min_new_tokens)


def _check():
    torch.manual_seed(0)
    for name in ("decoder", "encdec", "hybrid", "efficient"):
        model = build_model(name, vocab_size=VOCAB_SIZE)
        total = count_parameters(model)
        print(name, f"{total / 1e6:.2f}M")
        model.train()
        idx = torch.randint(1, 100, (2, 32))
        _logits, loss = model(idx, labels=idx)
        loss.backward()
        model.eval()
        produced = generate(model, idx[:, :8], max_new_tokens=3, temperature=0.0, top_p=1.0, eos_id=None, pad_id=0)
        assert len(produced) == 2
        assert all(len(row) == 3 for row in produced)
    print("phase1 model check passed")


In [ ]:
_register('phase1_models')


In [ ]:
"""Phase 2 legal named-entity recognition, starting from the Phase 1 models.

The score is strict macro-F1. A span counts only when the start character,
the end character, and the label all match. The previous draft of this file
trained a word BiLSTM and stopped at 400 words, so entities later in a
judgment were never predicted. This version:

  * loads one of the four Phase 1 models and adds a token-classification head
  * aligns the Phase 1 byte-level BPE to character offsets
  * slides a window across the whole document, including every test id
  * trains with full fine-tuning, LoRA, LoRMA+, or knowledge distillation

LoRMA+ is the multiplicative adapter from Bihany, Patel, and Modi (2025):

    h = (I + (alpha / r) B A) W x

with B initialised at zero, so the first step matches the Phase 1 model.

The four required submissions, each from a different Phase 1 architecture:

    python phase2_lner.py --arch efficient --method lorma
    python phase2_lner.py --arch decoder --method full
    python phase2_lner.py --arch hybrid --method lora
    python phase2_lner.py --arch encdec --method distill --teacher <efficient checkpoint>

Official deadline: 11 October 2026, 11:59 PM.
"""

import argparse
import json
import os
import random
import shutil
import zipfile

import torch
import torch.nn as nn
import torch.nn.functional as F

from legal_bpe import LegalBPE
from phase1_models import EncoderDecoderLM, build_model


SEED = 42
LABELS = [
    "APP",
    "RESP",
    "A.COUNSEL",
    "R.COUNSEL",
    "JUDGE",
    "WIT",
    "AUTH",
    "COURT",
    "STAT",
    "PREC",
    "DATE",
    "CASENO",
]
LOCAL = r"C:\Users\Anshuman Raj\CS781"
DEFAULT_WEIGHTS = {
    "efficient": os.path.join(LOCAL, r"kaggle_efficient2\cs781_phase1_efficient\model.pt"),
    "decoder": os.path.join(LOCAL, r"local_weights\cs781_phase1_decoder\model.pt"),
    "hybrid": os.path.join(LOCAL, r"local_weights\cs781_phase1_hybrid\model.pt"),
    "encdec": os.path.join(LOCAL, r"encdec_run2\cs781_phase1_encdec\model.pt"),
}
DEFAULT_BPE = os.path.join(LOCAL, r"kaggle_efficient2\cs781_phase1_efficient\legal_bpe.json")
REPO = "Exploration-Lab/CS-781-Capstone"
DATA_DIR = None
PROPAGATE = False
_GENERIC = {
    "state",
    "court",
    "india",
    "union",
    "government",
    "appellant",
    "appellants",
    "respondent",
    "respondents",
    "petitioner",
    "petitioners",
    "others",
    "other",
    "ors",
    "anr",
    "the",
    "shri",
    "smt",
    "and",
}


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def fetch(name):
    if DATA_DIR:
        local = os.path.join(DATA_DIR, *name.split("/"))
        if os.path.isfile(local):
            print("local", name, flush=True)
            return local
    from huggingface_hub import hf_hub_download

    path = hf_hub_download(repo_id=REPO, filename=name, repo_type="dataset")
    print("got", name, flush=True)
    return path


def read_json(path):
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)


def load_fold(text_name, gold_name=None):
    texts = read_json(fetch(text_name))
    golds = read_json(fetch(gold_name)) if gold_name else {}
    rows = []
    for doc_id, text in texts.items():
        spans = []
        for item in golds.get(doc_id, []):
            start, end, label = int(item[0]), int(item[1]), str(item[2])
            if label in LABELS and 0 <= start < end <= len(text):
                spans.append((start, end, label))
        rows.append((str(doc_id), text, spans))
    return rows


def tag_names_of():
    names = ["O"]
    for label in LABELS:
        names.append("B-" + label)
        names.append("I-" + label)
    return names


def bio_tags(token_spans, entities):
    """Label each token with the entity that covers the most of its characters."""
    assigned = [-1] * len(token_spans)
    for index, (left, right) in enumerate(token_spans):
        best = -1
        best_overlap = 0
        for entity_index, (start, end, _label) in enumerate(entities):
            overlap = min(right, end) - max(left, start)
            if overlap > best_overlap:
                best_overlap = overlap
                best = entity_index
        if best_overlap > 0:
            assigned[index] = best
    tags = []
    for index, entity_index in enumerate(assigned):
        if entity_index < 0:
            tags.append("O")
            continue
        label = entities[entity_index][2]
        previous = assigned[index - 1] if index else -1
        prefix = "I-" if previous == entity_index else "B-"
        tags.append(prefix + label)
    return tags


def make_windows(ids, spans, tags, width, stride):
    rows = []
    start = 0
    while start < len(ids):
        stop = min(start + width, len(ids))
        rows.append(
            {
                "ids": ids[start:stop],
                "spans": spans[start:stop],
                "tags": tags[start:stop],
                "offset": start,
            }
        )
        if stop == len(ids):
            break
        start += stride
    return rows


def prepare_document(tokenizer, doc_id, text, entities, width, stride, tag_index):
    ids, spans = tokenizer.encode_with_offsets(text)
    tags = bio_tags(spans, entities)
    windows = make_windows(ids, spans, tags, width, stride)
    for window in windows:
        window["doc_id"] = doc_id
        window["target"] = [tag_index[tag] for tag in window["tags"]]
    return ids, spans, tags, windows


def boundary_report(tokenizer, rows):
    """How often a gold span lands exactly on token boundaries."""
    covered = 0
    total = 0
    tiled = 0
    shown = 0
    for _doc_id, text, entities in rows:
        ids, spans = tokenizer.encode_with_offsets(text)
        if spans and spans[0][0] == 0 and spans[-1][1] == len(text):
            if all(right == next_left for (_left, right), (next_left, _next_right) in zip(spans, spans[1:])):
                tiled += 1
        starts = {left for left, _right in spans}
        ends = {right for _left, right in spans}
        for start, end, label in entities:
            total += 1
            if start in starts and end in ends:
                covered += 1
            elif shown < 5:
                shown += 1
                print(
                    "unaligned",
                    label,
                    repr(text[start:end][:60]),
                    "around",
                    repr(text[max(0, start - 8) : end + 8][:80]),
                )
    print(
        f"documents tiled by tokens {tiled}/{len(rows)}  "
        f"gold spans on token boundaries {covered}/{total}",
        flush=True,
    )
    return covered, total


class LoRALinear(nn.Module):
    """Additive low-rank update. The base Phase 1 matrix stays frozen."""

    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base = base
        self.base.weight.requires_grad_(False)
        if self.base.bias is not None:
            self.base.bias.requires_grad_(False)
        out_features, in_features = base.weight.shape
        self.A = nn.Parameter(torch.randn(rank, in_features) * 0.02)
        self.B = nn.Parameter(torch.zeros(out_features, rank))
        self.scale = alpha / rank

    def forward(self, x):
        update = (x @ self.A.transpose(0, 1)) @ self.B.transpose(0, 1)
        return self.base(x) + self.scale * update


class LoRMALinear(nn.Module):
    """LoRMA+ : h = (I + (alpha / r) B A) W x, with B starting at zero."""

    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base = base
        self.base.weight.requires_grad_(False)
        if self.base.bias is not None:
            self.base.bias.requires_grad_(False)
        out_features, _in_features = base.weight.shape
        self.A = nn.Parameter(torch.randn(rank, out_features) * 0.02)
        self.B = nn.Parameter(torch.zeros(out_features, rank))
        self.scale = alpha / rank

    def forward(self, x):
        hidden = self.base(x)
        update = (hidden @ self.A.transpose(0, 1)) @ self.B.transpose(0, 1)
        return hidden + self.scale * update


def adapt_linears(model, kind, rank, alpha):
    wrapper = LoRMALinear if kind == "lorma" else LoRALinear
    count = 0
    for module in list(model.modules()):
        for name, child in list(module.named_children()):
            if isinstance(child, nn.Linear):
                setattr(module, name, wrapper(child, rank, alpha))
                count += 1
    return count


def enable_full_attention(model):
    """Let a causal Phase 1 model read tokens on both sides while tagging.

    NER needs the words after a name. The language-model weights stay as
    trained; only the attention mask used for this task is removed.
    """
    count = 0
    for module in model.modules():
        if getattr(module, "causal", None) is not None and hasattr(module, "bidirectional"):
            module.bidirectional = True
            count += 1
    return count


class ChainCRF(nn.Module):
    """Linear-chain CRF over BIO tags. Illegal transitions stay blocked."""

    def __init__(self, tag_names):
        super().__init__()
        n_tags = len(tag_names)
        self.transitions = nn.Parameter(torch.zeros(n_tags, n_tags))
        allowed = torch.zeros(n_tags, n_tags, dtype=torch.bool)
        for src, src_name in enumerate(tag_names):
            for dst, dst_name in enumerate(tag_names):
                if dst_name == "O" or dst_name.startswith("B-"):
                    allowed[src, dst] = True
                else:
                    label = dst_name[2:]
                    allowed[src, dst] = src_name in ("B-" + label, "I-" + label)
        self.register_buffer("allowed", allowed)
        self.register_buffer("score_bias", torch.zeros(n_tags))
        with torch.no_grad():
            self.transitions.masked_fill_(~allowed, -10.0)

    def transition(self):
        return self.transitions.masked_fill(~self.allowed, -1e4)

    def _emit(self, emissions):
        return emissions + self.score_bias.view(1, 1, -1)

    def nll(self, emissions, tags, mask):
        emissions = self._emit(emissions)
        safe = tags.clamp(min=0)
        gold = self._path_score(emissions, safe, mask)
        logz = self._partition(emissions, mask)
        lengths = mask.float().sum(dim=1).clamp(min=1)
        return ((logz - gold) / lengths).mean()

    def _path_score(self, emissions, tags, mask):
        trans = self.transition()
        score = emissions[:, 0].gather(1, tags[:, 0:1]).squeeze(1)
        for step in range(1, emissions.size(1)):
            jump = trans[tags[:, step - 1], tags[:, step]]
            token = emissions[:, step].gather(1, tags[:, step : step + 1]).squeeze(1)
            score = score + (jump + token) * mask[:, step].float()
        return score

    def _partition(self, emissions, mask):
        trans = self.transition()
        alpha = emissions[:, 0]
        for step in range(1, emissions.size(1)):
            scores = alpha.unsqueeze(2) + trans.unsqueeze(0) + emissions[:, step].unsqueeze(1)
            nxt = torch.logsumexp(scores, dim=1)
            active = mask[:, step].unsqueeze(1)
            alpha = torch.where(active, nxt, alpha)
        return torch.logsumexp(alpha, dim=-1)

    def decode(self, emissions, mask):
        emissions = self._emit(emissions)
        trans = self.transition()
        batch, steps, n_tags = emissions.shape
        back = torch.zeros(batch, steps, n_tags, dtype=torch.long, device=emissions.device)
        alpha = emissions[:, 0]
        for step in range(1, steps):
            scores = alpha.unsqueeze(2) + trans.unsqueeze(0)
            best, index = scores.max(dim=1)
            nxt = best + emissions[:, step]
            active = mask[:, step].unsqueeze(1)
            alpha = torch.where(active, nxt, alpha)
            back[:, step] = index
        lengths = mask.long().sum(dim=1).clamp(min=1)
        tags = torch.zeros(batch, steps, dtype=torch.long, device=emissions.device)
        chosen = alpha.argmax(dim=-1)
        for row in range(batch):
            pos = int(lengths[row]) - 1
            tags[row, pos] = chosen[row]
            for step in range(pos, 0, -1):
                chosen[row] = back[row, step, tags[row, step]]
                tags[row, step - 1] = chosen[row]
        return tags


def hidden_states(model, token_ids, pad_mask):
    """Contextual vector for every token, before the language-model head."""
    if isinstance(model, EncoderDecoderLM):
        return model.encode(token_ids, pad_mask)
    hidden = model.token_plus_position(token_ids)
    hidden = model.run_blocks(model.blocks, hidden, pad_mask)
    return model.norm(hidden)


class TokenTagger(nn.Module):
    def __init__(self, backbone, n_tags, tag_names=None, use_crf=False):
        super().__init__()
        self.backbone = backbone
        width = backbone.embed.weight.size(1)
        self.drop = nn.Dropout(0.1)
        self.head = nn.Linear(width, n_tags)
        nn.init.zeros_(self.head.bias)
        self.head.bias.data[0] = 2.0
        self.crf = ChainCRF(tag_names) if use_crf else None

    def forward(self, token_ids, pad_mask):
        hidden = hidden_states(self.backbone, token_ids, pad_mask)
        return self.head(self.drop(hidden))


def build_tagger(arch, method, vocab_size, n_tags, weights_path, rank, alpha, tag_names=None, use_crf=False):
    backbone = build_model(arch, vocab_size=vocab_size)
    state = torch.load(weights_path, map_location="cpu", weights_only=True)
    backbone.load_state_dict(state)
    if method == "full":
        for parameter in backbone.parameters():
            parameter.requires_grad_(True)
        # NER uses encode() only. The causal decoder is unused, and its
        # gradients do not fit in 6GB next to a wider bidirectional window.
        if arch == "encdec":
            for parameter in backbone.decoder.parameters():
                parameter.requires_grad_(False)
    else:
        kind = "lorma" if method == "lorma" else "lora"
        wrapped = adapt_linears(backbone, kind, rank, alpha)
        print(f"wrapped {wrapped} linear layers with {kind}", flush=True)
    tagger = TokenTagger(backbone, n_tags, tag_names=tag_names, use_crf=use_crf)
    trainable = sum(parameter.numel() for parameter in tagger.parameters() if parameter.requires_grad)
    total = sum(parameter.numel() for parameter in tagger.parameters())
    print(f"parameters trainable {trainable / 1e6:.2f}M  total {total / 1e6:.2f}M", flush=True)
    return tagger


def class_weights(windows, n_tags, cap):
    counts = torch.ones(n_tags)
    for window in windows:
        for tag in window["target"]:
            counts[tag] += 1
    weights = 1.0 / counts.sqrt()
    weights = weights * (n_tags / weights.sum())
    weights = torch.clamp(weights, max=float(weights.median() * cap))
    return weights


def run_epoch(tagger, windows, optimizer, weights, device, train, batch_size, method, teacher=None, scaler=None):
    if train:
        tagger.train()
        # Frozen Phase 1 layers stay in eval so their dropout does not
        # scramble a 70-document training set. Adapter weights still learn.
        if method != "full":
            tagger.backbone.eval()
    else:
        tagger.eval()
    order = list(windows)
    if train:
        random.shuffle(order)
    total = 0.0
    seen = 0
    step = 0
    optimizer_was_used = optimizer is not None
    if optimizer_was_used:
        optimizer.zero_grad(set_to_none=True)
    for start in range(0, len(order), batch_size):
        chunk = order[start : start + batch_size]
        width = max(len(item["ids"]) for item in chunk)
        token_ids = torch.zeros(len(chunk), width, dtype=torch.long, device=device)
        targets = torch.full((len(chunk), width), -100, dtype=torch.long, device=device)
        pad_mask = torch.zeros(len(chunk), width, dtype=torch.bool, device=device)
        for row, item in enumerate(chunk):
            length = len(item["ids"])
            token_ids[row, :length] = torch.tensor(item["ids"], device=device)
            targets[row, :length] = torch.tensor(item["target"], device=device)
            pad_mask[row, :length] = True
        with torch.set_grad_enabled(train):
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                logits = tagger(token_ids, pad_mask).float()
                if tagger.crf is None:
                    loss = F.cross_entropy(
                        logits.reshape(-1, logits.size(-1)),
                        targets.reshape(-1),
                        weight=weights,
                        ignore_index=-100,
                    )
                else:
                    with torch.autocast(device_type=device.type, enabled=False):
                        loss = tagger.crf.nll(logits.float(), targets, pad_mask)
                if teacher is not None:
                    with torch.no_grad():
                        teacher_logits = teacher(token_ids, pad_mask)
                    temperature = 2.0
                    student_log = F.log_softmax(logits / temperature, dim=-1)
                    teacher_prob = F.softmax(teacher_logits / temperature, dim=-1)
                    kl = F.kl_div(student_log, teacher_prob, reduction="none").sum(dim=-1)
                    active = targets.ne(-100)
                    kl = (kl * active).sum() / active.sum().clamp(min=1)
                    loss = loss + (temperature * temperature) * kl
        if train:
            scaled = loss / 1.0
            if scaler is not None:
                scaler.scale(scaled).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(tagger.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                scaled.backward()
                torch.nn.utils.clip_grad_norm_(tagger.parameters(), 1.0)
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)
        total += float(loss.detach()) * len(chunk)
        seen += len(chunk)
        step += 1
        if train and step % 50 == 0:
            print(f"  step {step} loss {total / seen:.4f}", flush=True)
    return total / max(1, seen)


def trim_span(text, start, end):
    """Drop punctuation that is glued to a token but is outside the entity.

    The BPE often keeps the comma or sentence period on the last word.
    About 97% of the gold spans are an exact token span once those
    characters are removed. Periods that belong to 'v.' or 'Ltd.' stay.
    """
    while start < end and text[end - 1] in ",;:":
        end -= 1
    while start < end and text[start] == " ":
        start += 1
    if end - start >= 2 and text[end - 1] == "." and text[end - 2].isalnum():
        word = text[start:end]
        keep = word.endswith((" v.", " vs.", " No.", " Ltd.", " Pvt.", " Dr.", " Mr.", " Ms."))
        if not keep:
            end -= 1
    return start, end


def clean_edges(text, spans):
    """Fix citation edges that the tagger consistently gets one or two characters off.

    Held-out judgments keep the period on Ors., Anr. and Cr.P.C., and they
    leave out a glued footnote number, a closing parenthesis, and the words
    'reported in'. Party names are left alone: Ltd. is not a reliable period.
    """
    fixed = []
    for start, end, label in spans:
        if end > start and text[start:end].endswith(")."):
            end -= 2
        while end > start and text[end - 1] in ")]":
            end -= 1
        while end > start + 1 and text[end - 1].isdigit() and text[end - 2] in ".)":
            if end - start >= 5 and text[end - 5 : end - 1].isdigit():
                break
            end -= 1
        folded = text[start:end].casefold()
        marker = folded.find(" reported in ")
        if marker > 0:
            end = start + marker
        if label in ("PREC", "STAT", "CASENO") and end < len(text) and text[end] == ".":
            if text[start:end].endswith(("Ors", "Anr", "Cr.P.C", "I.P.C", "C.P.C")):
                end += 1
        if end > start:
            fixed.append((start, end, label))
    unique = []
    seen = set()
    for span in fixed:
        if span not in seen:
            seen.add(span)
            unique.append(span)
    return unique


def spans_from_tags(text, token_spans, tags, probabilities, thresholds):
    found = []
    index = 0
    while index < len(tags):
        tag = tags[index]
        if not tag.startswith("B-") and not tag.startswith("I-"):
            index += 1
            continue
        label = tag[2:]
        if probabilities[index] < thresholds.get(label, 0.0):
            index += 1
            continue
        cursor = index + 1
        while cursor < len(tags) and tags[cursor] == "I-" + label:
            if probabilities[cursor] < thresholds.get(label, 0.0):
                break
            cursor += 1
        start, end = trim_span(text, token_spans[index][0], token_spans[cursor - 1][1])
        if end > start:
            found.append((start, end, label))
        index = cursor
    # The evaluator rejects a repeated span, so keep the first copy only.
    unique = []
    seen = set()
    for span in found:
        if span not in seen:
            seen.add(span)
            unique.append(span)
    return unique


def propagate_spans(text, spans):
    """Copy a name the model already found onto later exact mentions.

    About two thirds of the gold appellant and respondent spans are the same
    string repeated in one judgment. A window tagger often hits the name once
    and misses the copies. Only labels that are names are copied, and a copy
    that sits inside a span already predicted is left alone.
    """
    occupied = [(start, end) for start, end, _label in spans]
    added = []

    def overlaps(start, end):
        return any(not (end <= left or start >= right) for left, right in occupied)

    def boundary(index):
        return index <= 0 or index >= len(text) or not text[index].isalnum()

    # Respondent, witness, and counsel copies added more wrong spans than
    # right ones on the held-out judgments. Appellant names did the opposite.
    minimum = {"APP": 8}
    folded = text.casefold()
    seen = set()
    for start, end, label in spans:
        if label not in minimum:
            continue
        raw = " ".join(text[start:end].split())
        key = raw.casefold()
        if len(key) < minimum[label] or key in _GENERIC or (label, key) in seen:
            continue
        seen.add((label, key))
        needle = key
        cursor = 0
        while True:
            at = folded.find(needle, cursor)
            if at < 0:
                break
            stop = at + len(needle)
            cursor = at + 1
            if not boundary(at - 1) or not boundary(stop):
                continue
            if overlaps(at, stop):
                continue
            tail = folded[stop : stop + 8]
            if label in ("APP", "RESP") and (
                tail.startswith(" v.") or tail.startswith(" vs") or tail.startswith(" versus")
            ):
                continue
            added.append((at, stop, label))
            occupied.append((at, stop))
    return spans + added


def predict_document(tagger, tokenizer, text, width, stride, tag_names, thresholds, device):
    ids, spans = tokenizer.encode_with_offsets(text)
    if not ids:
        return []
    summed = torch.zeros(len(ids), len(tag_names))
    counts = torch.zeros(len(ids), 1)
    tagger.eval()
    for window in make_windows(ids, spans, ["O"] * len(ids), width, stride):
        token_ids = torch.tensor([window["ids"]], dtype=torch.long, device=device)
        pad_mask = torch.ones(1, len(window["ids"]), dtype=torch.bool, device=device)
        with torch.no_grad():
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                logits = tagger(token_ids, pad_mask)[0].float().cpu()
        begin = window["offset"]
        summed[begin : begin + len(window["ids"])] += logits
        counts[begin : begin + len(window["ids"])] += 1
    averaged = summed / counts.clamp(min=1)
    if tagger.crf is None:
        probabilities, chosen = averaged.softmax(dim=-1).max(dim=-1)
        chosen = chosen.tolist()
        probabilities = probabilities.tolist()
    else:
        device = next(tagger.parameters()).device
        emissions = averaged.unsqueeze(0).to(device)
        mask = torch.ones(1, averaged.size(0), dtype=torch.bool, device=device)
        chosen = tagger.crf.decode(emissions, mask)[0].cpu().tolist()
        probabilities = averaged.softmax(dim=-1).gather(
            1, torch.tensor(chosen).unsqueeze(1)
        ).squeeze(1).tolist()
    tags = [tag_names[index] for index in chosen]
    found = spans_from_tags(text, spans, tags, probabilities, thresholds)
    if PROPAGATE:
        found = propagate_spans(text, found)
    return clean_edges(text, found)


def macro_f1(gold_rows, predictions):
    stats = {label: [0, 0, 0] for label in LABELS}
    for doc_id, _text, entities in gold_rows:
        gold = {(start, end, label) for start, end, label in entities}
        pred = set(tuple(span) for span in predictions.get(doc_id, []))
        for label in LABELS:
            gold_label = {span for span in gold if span[2] == label}
            pred_label = {span for span in pred if span[2] == label}
            stats[label][0] += len(gold_label & pred_label)
            stats[label][1] += len(pred_label - gold_label)
            stats[label][2] += len(gold_label - pred_label)
    f1s = []
    lines = []
    for label in LABELS:
        true_pos, false_pos, false_neg = stats[label]
        precision = true_pos / (true_pos + false_pos) if true_pos + false_pos else 0.0
        recall = true_pos / (true_pos + false_neg) if true_pos + false_neg else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        f1s.append(f1)
        lines.append(f"{label}: P={precision:.3f} R={recall:.3f} F1={f1:.3f}")
    return sum(f1s) / len(f1s), lines


def score_split(tagger, tokenizer, rows, width, stride, tag_names, thresholds, device):
    predictions = {}
    for index, (doc_id, text, _entities) in enumerate(rows):
        predictions[doc_id] = predict_document(
            tagger, tokenizer, text, width, stride, tag_names, thresholds, device
        )
        if index % 5 == 0:
            print(f"  scored {index + 1}/{len(rows)}", flush=True)
    f1, lines = macro_f1(rows, predictions)
    return f1, lines, predictions


def tune_thresholds(tagger, tokenizer, rows, width, stride, tag_names, device):
    """Pick one minimum probability per label on the held-out documents."""
    _f1, _lines, predictions = score_split(
        tagger, tokenizer, rows, width, stride, tag_names, {label: 0.0 for label in LABELS}, device
    )
    # Re-score from cached spans is not available, so search a short grid by
    # re-decoding is too expensive. The probabilities are already inside
    # predict_document. A second pass with a shared threshold is the stable
    # choice on only a handful of dev documents.
    best_f1 = _f1
    best = {label: 0.0 for label in LABELS}
    print(f"threshold 0.00  macro-F1 {best_f1:.4f}", flush=True)
    for minimum in (0.15, 0.30, 0.45, 0.60, 0.75):
        trial = {label: minimum for label in LABELS}
        f1, _lines, _predictions = score_split(
            tagger, tokenizer, rows, width, stride, tag_names, trial, device
        )
        print(f"threshold {minimum:.2f}  macro-F1 {f1:.4f}", flush=True)
        if f1 > best_f1:
            best_f1 = f1
            best = trial
    return best, best_f1


def split_documents(fold1, fold2, dev_count):
    ordered = sorted(fold2, key=lambda row: row[0])
    dev_rows = ordered[-dev_count:]
    dev_ids = {row[0] for row in dev_rows}
    train_rows = list(fold1) + [row for row in fold2 if row[0] not in dev_ids]
    return train_rows, dev_rows


def collect_windows(tokenizer, rows, width, stride, tag_index):
    windows = []
    for doc_id, text, entities in rows:
        _ids, _spans, _tags, doc_windows = prepare_document(
            tokenizer, doc_id, text, entities, width, stride, tag_index
        )
        windows.extend(doc_windows)
    return windows


def save_checkpoint(path, tagger, args, tag_names, thresholds, dev_f1):
    torch.save(
        {
            "arch": args.arch,
            "method": args.method,
            "tag_names": tag_names,
            "thresholds": thresholds,
            "dev_f1": dev_f1,
            "model": tagger.state_dict(),
        },
        path,
    )
    print("saved", path, flush=True)


def write_submission(path, predictions):
    payload = {doc_id: [list(span) for span in spans] for doc_id, spans in predictions.items()}
    folder = os.path.dirname(path)
    os.makedirs(folder, exist_ok=True)
    pred_path = os.path.join(folder, "prediction.json")
    with open(pred_path, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, ensure_ascii=False)
    with zipfile.ZipFile(path, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
        bundle.write(pred_path, arcname="prediction.json")
    print("zip", path, "docs", len(payload), flush=True)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--arch", required=True, choices=["decoder", "encdec", "hybrid", "efficient"])
    parser.add_argument("--method", required=True, choices=["full", "lora", "lorma", "distill"])
    parser.add_argument("--weights", default=None)
    parser.add_argument("--bpe", default=DEFAULT_BPE)
    parser.add_argument("--teacher", default=None, help="Checkpoint from a larger Phase 2 model.")
    parser.add_argument("--teacher_arch", default="efficient")
    parser.add_argument("--teacher_method", default="lorma")
    parser.add_argument("--teacher_weights", default=None)
    parser.add_argument("--out", default=None)
    parser.add_argument("--epochs", type=int, default=8)
    parser.add_argument("--batch", type=int, default=0)
    parser.add_argument("--width", type=int, default=0)
    parser.add_argument("--stride", type=int, default=0)
    parser.add_argument("--rank", type=int, default=8)
    parser.add_argument("--alpha", type=float, default=16.0)
    parser.add_argument("--lr", type=float, default=0.0)
    parser.add_argument("--head_lr", type=float, default=0.0, help="Learning rate for the tag head and CRF.")
    parser.add_argument("--dev_count", type=int, default=8)
    parser.add_argument("--weight_cap", type=float, default=5.0)
    parser.add_argument("--crf", action="store_true")
    parser.add_argument("--bidirectional", action="store_true")
    parser.add_argument("--data_dir", default=None, help="Local Phase_2_LNER folder, skipping Hugging Face.")
    parser.add_argument("--check", action="store_true")
    parser.add_argument("--resume", default=None, help="Phase 2 checkpoint to continue from.")
    parser.add_argument("--unfreeze", action="store_true", help="Train the Phase 1 weights under the adapter.")
    parser.add_argument("--score_only", action="store_true", help="Score the resume checkpoint and write a zip.")
    parser.add_argument("--seed", type=int, default=SEED)
    args = parser.parse_args()
    global DATA_DIR
    DATA_DIR = args.data_dir

    if args.method == "full":
        args.batch = args.batch or 1
        args.width = args.width or 256
        args.stride = args.stride or 128
        args.lr = args.lr or 2e-5
    else:
        args.batch = args.batch or 2
        args.width = args.width or 384
        args.stride = args.stride or 192
        args.lr = args.lr or 1e-4
    if args.weights is None:
        args.weights = DEFAULT_WEIGHTS[args.arch]
    if args.out is None:
        args.out = os.path.join(LOCAL, "phase2_runs", f"{args.arch}_{args.method}")
    os.makedirs(args.out, exist_ok=True)

    set_seed(args.seed)
    print("seed", args.seed, flush=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device", device, flush=True)
    if not os.path.isfile(args.weights):
        raise SystemExit(f"missing Phase 1 weights: {args.weights}")
    if not os.path.isfile(args.bpe):
        raise SystemExit(f"missing BPE: {args.bpe}")

    tokenizer = LegalBPE.load(args.bpe)
    fold1 = load_fold("Phase_2_LNER/Train/fold1-text.json", "Phase_2_LNER/Train/fold1-gold.json")
    fold2 = load_fold("Phase_2_LNER/Train/fold2-text.json", "Phase_2_LNER/Train/fold2-gold.json")
    print("fold1", len(fold1), "fold2", len(fold2), flush=True)
    covered, total = boundary_report(tokenizer, fold1 + fold2)
    if total:
        print(f"exact token-boundary rate {covered / total:.3f}", flush=True)
    if args.check:
        return

    train_rows, dev_rows = split_documents(fold1, fold2, args.dev_count)
    print("train docs", len(train_rows), "dev docs", [row[0] for row in dev_rows], flush=True)
    names = tag_names_of()
    tag_index = {name: index for index, name in enumerate(names)}
    train_windows = collect_windows(tokenizer, train_rows, args.width, args.stride, tag_index)
    print("train windows", len(train_windows), "width", args.width, "stride", args.stride, flush=True)

    tagger = build_tagger(
        args.arch, args.method, len(tokenizer), len(names), args.weights, args.rank, args.alpha, names, args.crf
    )
    if args.bidirectional:
        opened = enable_full_attention(tagger.backbone)
        print("full attention layers", opened, flush=True)
    best_path = os.path.join(args.out, "best.pt")
    best_f1 = -1.0
    if args.resume:
        resumed = torch.load(args.resume, map_location="cpu", weights_only=True)
        tagger.load_state_dict(resumed["model"])
        best_f1 = float(resumed.get("dev_f1", -1.0))
        print("resumed", args.resume, "dev macro-F1", round(best_f1, 4), flush=True)
    if args.unfreeze:
        opened = 0
        for module in tagger.modules():
            if isinstance(module, (LoRMALinear, LoRALinear)):
                module.base.weight.requires_grad_(True)
                if module.base.bias is not None:
                    module.base.bias.requires_grad_(True)
                opened += 1
        print("unfroze", opened, "adapter base matrices", flush=True)
    tagger = tagger.to(device)
    teacher = None
    if args.method == "distill":
        if not args.teacher:
            raise SystemExit("distill needs --teacher pointing at a Phase 2 checkpoint")
        teacher_weights = args.teacher_weights or DEFAULT_WEIGHTS[args.teacher_arch]
        checkpoint = torch.load(args.teacher, map_location="cpu", weights_only=True)
        teacher_state = checkpoint["model"]
        teacher_crf = any(key.startswith("crf.") for key in teacher_state)
        teacher = build_tagger(
            args.teacher_arch,
            args.teacher_method,
            len(tokenizer),
            len(names),
            teacher_weights,
            args.rank,
            args.alpha,
            names,
            teacher_crf,
        )
        teacher.load_state_dict(teacher_state)
        if args.bidirectional:
            opened = enable_full_attention(teacher.backbone)
            print("teacher full attention layers", opened, flush=True)
        teacher = teacher.to(device).eval()
        for parameter in teacher.parameters():
            parameter.requires_grad_(False)

    weights = class_weights(train_windows, len(names), args.weight_cap).to(device)
    if tagger.crf is not None:
        tagger.crf.score_bias.copy_(weights.log().to(tagger.crf.score_bias.device))
    if args.score_only:
        global PROPAGATE
        zero = {label: 0.0 for label in LABELS}
        PROPAGATE = False
        base_f1, base_lines, _base_pred = score_split(
            tagger, tokenizer, dev_rows, args.width, args.stride, names, zero, device
        )
        print(f"base dev macro-F1 {base_f1:.4f}", flush=True)
        print("\n".join(base_lines), flush=True)
        PROPAGATE = True
        prop_f1, prop_lines, _prop_pred = score_split(
            tagger, tokenizer, dev_rows, args.width, args.stride, names, zero, device
        )
        print(f"propagate dev macro-F1 {prop_f1:.4f}", flush=True)
        print("\n".join(prop_lines), flush=True)
        PROPAGATE = prop_f1 > base_f1 + 1e-4
        print("zip uses propagate" if PROPAGATE else "zip uses the model spans only", flush=True)
        test_rows = load_fold("Phase_2_LNER/Test/fold3-text.json")
        predictions = {}
        for index, (doc_id, text, _entities) in enumerate(test_rows):
            predictions[doc_id] = predict_document(
                tagger, tokenizer, text, args.width, args.stride, names, zero, device
            )
            print(f"test {index + 1}/{len(test_rows)} id {doc_id} spans {len(predictions[doc_id])}", flush=True)
        write_submission(os.path.join(args.out, "submission.zip"), predictions)
        return
    head_params = []
    backbone_params = []
    for name, parameter in tagger.named_parameters():
        if not parameter.requires_grad:
            continue
        if name.startswith("head.") or name.startswith("crf."):
            head_params.append(parameter)
        else:
            backbone_params.append(parameter)
    if args.head_lr:
        optimizer = torch.optim.AdamW(
            [
                {"params": backbone_params, "lr": args.lr},
                {"params": head_params, "lr": args.head_lr},
            ],
            weight_decay=0.01,
        )
        print(f"lr backbone {args.lr} head {args.head_lr}", flush=True)
    else:
        optimizer = torch.optim.AdamW(backbone_params + head_params, lr=args.lr, weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    if args.resume and not os.path.isfile(best_path):
        shutil.copyfile(args.resume, best_path)

    patience = 0
    for epoch in range(args.epochs):
        loss = run_epoch(
            tagger, train_windows, optimizer, weights, device, True, args.batch, args.method, teacher, scaler
        )
        print(f"epoch {epoch + 1} train loss {loss:.4f}", flush=True)
        f1, lines, _predictions = score_split(
            tagger, tokenizer, dev_rows, args.width, args.stride, names, {label: 0.0 for label in LABELS}, device
        )
        print(f"epoch {epoch + 1} dev macro-F1 {f1:.4f}", flush=True)
        print("\n".join(lines), flush=True)
        if f1 > best_f1:
            best_f1 = f1
            patience = 0
            save_checkpoint(best_path, tagger, args, names, {label: 0.0 for label in LABELS}, f1)
        else:
            patience += 1
            if patience >= 3:
                print("early stop", flush=True)
                break

    checkpoint = torch.load(best_path, map_location="cpu", weights_only=True)
    tagger.load_state_dict(checkpoint["model"])
    tagger.to(device)
    thresholds, tuned_f1 = tune_thresholds(
        tagger, tokenizer, dev_rows, args.width, args.stride, names, device
    )
    print("chosen thresholds", thresholds, "dev macro-F1", round(tuned_f1, 4), flush=True)
    save_checkpoint(best_path, tagger, args, names, thresholds, tuned_f1)

    test_rows = load_fold("Phase_2_LNER/Test/fold3-text.json")
    predictions = {}
    for index, (doc_id, text, _entities) in enumerate(test_rows):
        predictions[doc_id] = predict_document(
            tagger, tokenizer, text, args.width, args.stride, names, thresholds, device
        )
        print(f"test {index + 1}/{len(test_rows)} id {doc_id} spans {len(predictions[doc_id])}", flush=True)
    write_submission(os.path.join(args.out, "submission.zip"), predictions)


In [ ]:
_register('phase2_lner')


## 3. Role tagger (this is the selected model)


In [ ]:
"""Phase 3 rhetorical roles, built on the four Phase 1 models.

Every sentence is read by a Phase 1 model (own byte-level BPE, own weights)
and mean-pooled into one vector. A BiLSTM reads the sentence vectors of a
whole judgment in order, and a linear-chain CRF picks one of the 13 roles for
each sentence. The score is macro-F1 over all 13 roles, so a role that never
gets predicted counts as zero.

Adaptation of the Phase 1 model (--method):
    frozen   Phase 1 weights fixed; sentence vectors cached once. Baseline.
    lorma    LoRMA+  h = (I + (alpha/r) B A) W x
    lora     LoRA    h = W x + (alpha/r) B A x
    rslora   LoRA with the rank-stabilised scale alpha/sqrt(r)
    ia3      IA3: learned vectors rescale keys, values, and the
             feed-forward activation; nothing else in the Phase 1 model moves
    distill  a student (--student_method, LoRA by default) learns from the
             gold roles and from the role scores of a trained teacher
             (--teacher), usually the efficient LoRMA run

    python phase3_rr_own.py --arch efficient --method lorma --out <folder>
    python phase3_rr_own.py --arch encdec --method distill --teacher <efficient lorma>/best.pt --out <folder>

The best epoch by held-out macro-F1 writes <out>/submission.zip, which holds
prediction.json: {document id: [one role per sentence]}.
"""

import argparse
import json
import math
import os
import random
import time
import zipfile

import torch
import torch.nn as nn
import torch.nn.functional as F

from legal_bpe import LegalBPE
from phase1_models import build_model
from phase2_lner import LoRALinear, adapt_linears, enable_full_attention, hidden_states

LOCAL = r"C:\Users\Anshuman Raj\CS781"
DEFAULT_WEIGHTS = {
    "efficient": os.path.join(LOCAL, r"kaggle_efficient2\cs781_phase1_efficient\model.pt"),
    "decoder": os.path.join(LOCAL, r"local_weights\cs781_phase1_decoder\model.pt"),
    "hybrid": os.path.join(LOCAL, r"local_weights\cs781_phase1_hybrid\model.pt"),
    "encdec": os.path.join(LOCAL, r"encdec_run2\cs781_phase1_encdec\model.pt"),
}
DEFAULT_BPE = os.path.join(LOCAL, r"kaggle_efficient2\cs781_phase1_efficient\legal_bpe.json")
DEFAULT_DATA = os.path.join(LOCAL, "phase3data")
TRAIN_FILES = ("Train/CL_train.json", "Train/IT_train.json")
DEV_FILES = ("Train/CL_dev.json", "Train/IT_dev.json")
TEST_FILES = ("Test/CL_test.json", "Test/IT_test.json")
DEFAULT_LR = {"lorma": 2e-4, "lora": 2e-4, "rslora": 2e-4, "ia3": 3e-3, "frozen": 0.0}


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def read_vocab(data_dir):
    path = os.path.join(data_dir, "Phase_3_RR", "Train", "label_vocab.json")
    with open(path, encoding="utf-8") as handle:
        mapping = json.load(handle)
    return [label for label, _index in sorted(mapping.items(), key=lambda item: item[1])]


def load_docs(data_dir, names):
    docs = []
    for name in names:
        path = os.path.join(data_dir, "Phase_3_RR", *name.split("/"))
        with open(path, encoding="utf-8") as handle:
            data = json.load(handle)
        domain = "CL" if "/CL_" in name else "IT"
        if isinstance(data, dict):
            for doc_id, sentences in data.items():
                docs.append({"id": doc_id, "domain": domain, "text": list(sentences), "labels": None})
        else:
            for item in data:
                docs.append(
                    {"id": str(item["id"]), "domain": domain, "text": list(item["text"]), "labels": list(item["labels"])}
                )
    return docs


def tokenize_docs(tokenizer, docs, max_tokens):
    for doc in docs:
        doc["ids"] = [tokenizer.encode(sentence)[:max_tokens] or [tokenizer.unk_token_id] for sentence in doc["text"]]


class IA3Linear(nn.Module):
    """A frozen Phase 1 matrix with one learned scaling vector, starting at 1.

    On keys and values the vector scales the output. On the feed-forward
    down projection it scales the input, which is the activation after the
    non-linearity. For a fused query-key-value matrix the query part is kept
    at 1 through the mask.
    """

    def __init__(self, base, size, on_input, keep=None):
        super().__init__()
        self.base = base
        for parameter in base.parameters():
            parameter.requires_grad_(False)
        self.on_input = on_input
        self.scale = nn.Parameter(torch.ones(size))
        self.register_buffer("mask", torch.ones(size) if keep is None else keep)

    def factor(self):
        return 1 + self.mask * (self.scale - 1)

    def forward(self, x):
        if self.on_input:
            return self.base(x * self.factor().to(x.dtype))
        out = self.base(x)
        return out * self.factor().to(out.dtype)


def add_ia3(root):
    count = 0
    for module in list(root.modules()):
        qkv = getattr(module, "qkv", None)
        if isinstance(qkv, nn.Linear):
            width = qkv.out_features // 3
            keep = torch.zeros(qkv.out_features)
            keep[width:] = 1
            module.qkv = IA3Linear(qkv, qkv.out_features, False, keep)
            count += 1
        for name in ("k", "v"):
            layer = getattr(module, name, None)
            if isinstance(layer, nn.Linear):
                setattr(module, name, IA3Linear(layer, layer.out_features, False))
                count += 1
        down = getattr(module, "down", None)
        if isinstance(down, nn.Linear):
            module.down = IA3Linear(down, down.in_features, True)
            count += 1
    return count


class RoleCRF(nn.Module):
    """Linear-chain CRF over sentence roles. Any role may follow any role."""

    def __init__(self, n_labels):
        super().__init__()
        self.start = nn.Parameter(torch.zeros(n_labels))
        self.end = nn.Parameter(torch.zeros(n_labels))
        self.trans = nn.Parameter(torch.zeros(n_labels, n_labels))

    def nll(self, emissions, tags):
        steps = emissions.size(0)
        gold = self.start[tags[0]] + emissions[0, tags[0]] + self.end[tags[-1]]
        if steps > 1:
            gold = gold + self.trans[tags[:-1], tags[1:]].sum()
            gold = gold + emissions[1:].gather(1, tags[1:].unsqueeze(1)).sum()
        alpha = self.start + emissions[0]
        for step in range(1, steps):
            alpha = torch.logsumexp(alpha.unsqueeze(1) + self.trans + emissions[step].unsqueeze(0), dim=0)
        logz = torch.logsumexp(alpha + self.end, dim=0)
        return (logz - gold) / steps

    @torch.no_grad()
    def decode(self, emissions):
        steps = emissions.size(0)
        score = self.start + emissions[0]
        back = []
        for step in range(1, steps):
            best, index = (score.unsqueeze(1) + self.trans).max(dim=0)
            score = best + emissions[step]
            back.append(index)
        last = int((score + self.end).argmax())
        path = [last]
        for index in reversed(back):
            last = int(index[last])
            path.append(last)
        return path[::-1]


DOMAIN_INDEX = {"CL": 0, "IT": 1}


class RoleTagger(nn.Module):
    def __init__(self, backbone, n_labels, hidden=256, layers=1, domain_dim=0, attn_heads=0, buckets=20):
        super().__init__()
        self.backbone = backbone
        width = backbone.embed.weight.size(1)
        self.buckets = buckets
        self.domain_dim = domain_dim
        self.attn_heads = attn_heads
        self.position = nn.Embedding(buckets, 32)
        self.domain = nn.Embedding(2, domain_dim) if domain_dim else None
        self.drop = nn.Dropout(0.2)
        input_size = width + 32 + domain_dim
        self.norm = nn.LayerNorm(input_size)
        self.context = nn.LSTM(
            input_size, hidden, num_layers=layers, batch_first=True, bidirectional=True, dropout=0.1 if layers > 1 else 0.0
        )
        context_size = 2 * hidden
        if attn_heads:
            self.attn = nn.MultiheadAttention(context_size, attn_heads, dropout=0.1, batch_first=True)
            self.attn_norm = nn.LayerNorm(context_size)
        else:
            self.attn = None
        self.head = nn.Linear(context_size, n_labels)
        self.crf = RoleCRF(n_labels)

    def encode(self, token_ids, pad_mask):
        hidden = hidden_states(self.backbone, token_ids, pad_mask)
        weight = pad_mask.unsqueeze(-1).to(hidden.dtype)
        return (hidden * weight).sum(dim=1) / weight.sum(dim=1).clamp(min=1)

    def emissions(self, vectors, positions, domain="CL"):
        """vectors [sentences, width], positions [sentences] in [0, 1)."""
        bucket = (positions * self.buckets).long().clamp(0, self.buckets - 1)
        parts = [vectors.float(), self.position(bucket)]
        if self.domain is not None:
            index = torch.full((vectors.size(0),), DOMAIN_INDEX[domain], device=vectors.device, dtype=torch.long)
            parts.append(self.domain(index))
        features = self.norm(torch.cat(parts, dim=-1)).unsqueeze(0)
        context, _state = self.context(self.drop(features))
        if self.attn is not None:
            attended, _weights = self.attn(context, context, context, need_weights=False)
            context = self.attn_norm(context + self.drop(attended))
        return self.head(self.drop(context))[0]


def build_tagger(arch, method, weights, rank, alpha, n_labels, head_hidden=256, head_layers=1, domain_dim=0, attn_heads=0):
    backbone = build_model(arch, vocab_size=8192)
    backbone.load_state_dict(torch.load(weights, map_location="cpu", weights_only=True))
    enable_full_attention(backbone)
    for parameter in backbone.parameters():
        parameter.requires_grad_(False)
    # The encoder-decoder tags with its encoder only; the decoder never runs.
    blocks = backbone.encoder if arch == "encdec" else backbone.blocks
    if method in ("lorma", "lora", "rslora"):
        wrapped = adapt_linears(blocks, "lorma" if method == "lorma" else "lora", rank, alpha)
        if method == "rslora":
            for module in blocks.modules():
                if isinstance(module, LoRALinear):
                    module.scale = alpha / math.sqrt(rank)
        print(f"wrapped {wrapped} linear layers with {method}", flush=True)
    elif method == "ia3":
        print(f"wrapped {add_ia3(blocks)} linear layers with ia3", flush=True)
    elif method != "frozen":
        raise SystemExit(f"unknown method {method}")
    tagger = RoleTagger(backbone, n_labels, head_hidden, head_layers, domain_dim, attn_heads)
    tagger.adapter_keys = {
        name for name, parameter in tagger.named_parameters() if name.startswith("backbone.") and parameter.requires_grad
    }
    print(
        f"head hidden {head_hidden} layers {head_layers} domain_dim {domain_dim} attn_heads {attn_heads}",
        flush=True,
    )
    return tagger


def head_options(checkpoint_or_args):
    """Read head settings from a checkpoint dict or argparse namespace."""
    get = checkpoint_or_args.get if isinstance(checkpoint_or_args, dict) else lambda key, default=None: getattr(
        checkpoint_or_args, key, default
    )
    return {
        "head_hidden": int(get("head_hidden", 256) or 256),
        "head_layers": int(get("head_layers", 1) or 1),
        "domain_dim": int(get("domain_dim", 0) or 0),
        "attn_heads": int(get("attn_heads", 0) or 0),
    }


def backbone_mode(tagger, method, training):
    """Train mode turns on block checkpointing; dropout inside the Phase 1 model stays off."""
    if not training or method == "frozen":
        tagger.backbone.eval()
        return
    tagger.backbone.train()
    for module in tagger.backbone.modules():
        if isinstance(module, nn.Dropout):
            module.eval()


def pad_batch(rows, device):
    width = max(len(row) for row in rows)
    token_ids = torch.zeros(len(rows), width, dtype=torch.long)
    pad_mask = torch.zeros(len(rows), width, dtype=torch.bool)
    for index, row in enumerate(rows):
        token_ids[index, : len(row)] = torch.tensor(row)
        pad_mask[index, : len(row)] = True
    return token_ids.to(device), pad_mask.to(device)


def encode_sentences(tagger, rows, device, token_budget=8192):
    """Sentence vectors without gradients, batched by length."""
    order = sorted(range(len(rows)), key=lambda index: len(rows[index]))
    vectors = [None] * len(rows)
    start = 0
    while start < len(order):
        stop = start + 1
        while stop < len(order) and (stop - start + 1) * len(rows[order[stop]]) <= token_budget:
            stop += 1
        chunk = order[start:stop]
        token_ids, pad_mask = pad_batch([rows[index] for index in chunk], device)
        with torch.no_grad(), torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            pooled = tagger.encode(token_ids, pad_mask).float().cpu()
        for slot, index in enumerate(chunk):
            vectors[index] = pooled[slot]
        start = stop
    return torch.stack(vectors)


def positions_of(count, start=0, length=None):
    length = length if length is not None else count
    return torch.arange(start, start + length, dtype=torch.float) / max(1, count)


def doc_vectors(tagger, docs, device):
    tagger.backbone.eval()
    return {doc["id"]: encode_sentences(tagger, doc["ids"], device) for doc in docs}


def doc_emissions(tagger, docs, vectors, device):
    tagger.eval()
    out = {}
    with torch.no_grad():
        for doc in docs:
            sentences = vectors[doc["id"]].to(device)
            out[doc["id"]] = tagger.emissions(
                sentences, positions_of(len(doc["ids"])).to(device), doc["domain"]
            ).float()
    return out


def predict(tagger, docs, vectors, vocab, device):
    emissions = doc_emissions(tagger, docs, vectors, device)
    return {doc_id: [vocab[index] for index in tagger.crf.decode(scores)] for doc_id, scores in emissions.items()}


def macro_f1(docs, predictions, vocab):
    """Same rule as the course evaluator: all 13 roles, zero when undefined."""
    stats = {label: [0, 0, 0] for label in vocab}
    for doc in docs:
        for gold, guess in zip(doc["labels"], predictions[doc["id"]]):
            if gold == guess:
                stats[gold][0] += 1
            else:
                stats[guess][1] += 1
                stats[gold][2] += 1
    lines = []
    f1s = []
    for label in vocab:
        true_pos, false_pos, false_neg = stats[label]
        precision = true_pos / (true_pos + false_pos) if true_pos + false_pos else 0.0
        recall = true_pos / (true_pos + false_neg) if true_pos + false_neg else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        f1s.append(f1)
        lines.append(f"  {label:24s} P={precision:.3f} R={recall:.3f} F1={f1:.3f} gold={true_pos + false_neg}")
    return sum(f1s) / len(f1s), lines


def domain_scores(docs, predictions, vocab):
    parts = []
    for domain in ("CL", "IT"):
        subset = [doc for doc in docs if doc["domain"] == domain]
        if subset:
            parts.append(f"{domain} {macro_f1(subset, predictions, vocab)[0]:.4f}")
    return "  ".join(parts)


def class_weights(docs, vocab, cap):
    counts = torch.ones(len(vocab))
    index = {label: position for position, label in enumerate(vocab)}
    for doc in docs:
        for label in doc["labels"]:
            counts[index[label]] += 1
    weights = 1.0 / counts.sqrt()
    weights = weights * (len(vocab) / weights.sum())
    return torch.clamp(weights, max=float(weights.median() * cap))


def role_loss(tagger, emissions, tags, weights, ce_weight):
    loss = tagger.crf.nll(emissions.float(), tags)
    if ce_weight > 0:
        loss = loss + ce_weight * F.cross_entropy(emissions.float(), tags, weight=weights)
    return loss


def distill_loss(student, teacher, temperature):
    student_log = F.log_softmax(student.float() / temperature, dim=-1)
    teacher_prob = F.softmax(teacher.float() / temperature, dim=-1)
    return (temperature * temperature) * F.kl_div(student_log, teacher_prob, reduction="batchmean")


def training_chunks(docs, chunk):
    """Consecutive sentence chunks; the cut points shift every epoch."""
    pieces = []
    for doc in docs:
        count = len(doc["ids"])
        offset = random.randrange(chunk) if count > chunk else 0
        cuts = sorted({0, count, *range(offset, count, chunk)})
        for start, stop in zip(cuts, cuts[1:]):
            pieces.append((doc, start, stop))
    random.shuffle(pieces)
    return pieces


def trained_state(tagger):
    """Only what training changed: the head, the adapters, and their masks."""
    return {
        name: value
        for name, value in tagger.state_dict().items()
        if not name.startswith("backbone.") or name in tagger.adapter_keys or name.endswith(".mask")
    }


def load_trained(tagger, state):
    result = tagger.load_state_dict(state, strict=False)
    if result.unexpected_keys:
        raise SystemExit(f"checkpoint keys not in the model: {result.unexpected_keys[:5]}")
    missing = [name for name in result.missing_keys if not name.startswith("backbone.")]
    if missing:
        raise SystemExit(f"checkpoint is missing head keys: {missing[:5]}")


def save_checkpoint(path, tagger, args, method, vocab, dev_f1):
    torch.save(
        {
            "arch": args.arch,
            "method": method,
            "rank": args.rank,
            "alpha": args.alpha,
            "weights": args.weights,
            "vocab": vocab,
            "dev_f1": dev_f1,
            "head_hidden": args.head_hidden,
            "head_layers": args.head_layers,
            "domain_dim": args.domain_dim,
            "attn_heads": args.attn_heads,
            "model": trained_state(tagger),
        },
        path,
    )
    print("saved", path, f"dev macro-F1 {dev_f1:.4f}", flush=True)


def teacher_scores(args, docs, n_labels, device):
    """Role scores of the trained teacher for every training sentence."""
    checkpoint = torch.load(args.teacher, map_location="cpu", weights_only=False)
    weights = args.teacher_weights or checkpoint["weights"]
    print("teacher", checkpoint["arch"], checkpoint["method"], "dev", round(checkpoint["dev_f1"], 4), flush=True)
    teacher = build_tagger(
        checkpoint["arch"],
        checkpoint["method"],
        weights,
        checkpoint["rank"],
        checkpoint["alpha"],
        n_labels,
        **head_options(checkpoint),
    )
    load_trained(teacher, checkpoint["model"])
    teacher.to(device)
    scores = doc_emissions(teacher, docs, doc_vectors(teacher, docs, device), device)
    scores = {doc_id: value.cpu() for doc_id, value in scores.items()}
    del teacher
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return scores


def write_outputs(out_dir, predictions):
    pred_path = os.path.join(out_dir, "prediction.json")
    with open(pred_path, "w", encoding="utf-8") as handle:
        json.dump(predictions, handle, ensure_ascii=False)
    zip_path = os.path.join(out_dir, "submission.zip")
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
        bundle.write(pred_path, arcname="prediction.json")
    print("zip", zip_path, "docs", len(predictions), flush=True)


def write_dev_files(out_dir, dev_docs, predictions):
    """Files for a cross-check with evaluate_phase3.py."""
    for domain in ("CL", "IT"):
        truth = {doc["id"]: doc["labels"] for doc in dev_docs if doc["domain"] == domain}
        with open(os.path.join(out_dir, f"dev_truth_{domain.lower()}.json"), "w", encoding="utf-8") as handle:
            json.dump(truth, handle, ensure_ascii=False)
    with open(os.path.join(out_dir, "dev_prediction.json"), "w", encoding="utf-8") as handle:
        json.dump({doc["id"]: predictions[doc["id"]] for doc in dev_docs}, handle, ensure_ascii=False)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--arch", choices=("efficient", "decoder", "hybrid", "encdec"), default="efficient")
    parser.add_argument("--method", choices=("frozen", "lorma", "lora", "rslora", "ia3", "distill"), default="lorma")
    parser.add_argument("--student_method", choices=("lorma", "lora", "rslora", "ia3"), default="lora")
    parser.add_argument("--teacher", default=None, help="best.pt of a finished Phase 3 run.")
    parser.add_argument("--teacher_weights", default=None, help="Phase 1 weights of the teacher, if moved.")
    parser.add_argument("--kd_weight", type=float, default=0.5, help="Share of the loss taken by the teacher.")
    parser.add_argument("--kd_temp", type=float, default=2.0)
    parser.add_argument("--weights", default=None)
    parser.add_argument("--bpe", default=DEFAULT_BPE)
    parser.add_argument("--data_dir", default=DEFAULT_DATA)
    parser.add_argument("--out", required=True)
    parser.add_argument("--epochs", type=int, default=0, help="0 picks 40 for frozen and 8 otherwise.")
    parser.add_argument("--patience", type=int, default=0, help="0 picks 8 for frozen and 3 otherwise.")
    parser.add_argument("--lr", type=float, default=0.0, help="Adapter learning rate; 0 picks one per method.")
    parser.add_argument("--head_lr", type=float, default=1e-3, help="BiLSTM, head, and CRF.")
    parser.add_argument("--rank", type=int, default=8)
    parser.add_argument("--alpha", type=float, default=16.0)
    parser.add_argument("--chunk", type=int, default=32, help="Consecutive sentences per training step.")
    parser.add_argument("--train_tokens", type=int, default=96, help="Tokens kept per sentence while training.")
    parser.add_argument("--eval_tokens", type=int, default=160, help="Tokens kept per sentence at prediction.")
    parser.add_argument("--weight_cap", type=float, default=5.0)
    parser.add_argument("--ce_weight", type=float, default=1.0, help="Class-weighted cross-entropy added to the CRF loss.")
    parser.add_argument("--max_steps", type=int, default=0, help="Cap steps per epoch, for a quick check.")
    parser.add_argument(
        "--refit_from",
        default=None,
        help="best.pt of a finished run. Its adapters stay fixed; the BiLSTM and CRF retrain on whole judgments.",
    )
    parser.add_argument(
        "--adapters_from",
        default=None,
        help="best.pt whose LoRMA/IA3 keys are loaded into a freshly built head (CLI head_* flags apply).",
    )
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--fixed", action="store_true", help="Train all epochs and keep the last one, no dev selection.")
    parser.add_argument(
        "--include_dev", action="store_true", help="Also train on the labelled dev documents. Needs --fixed."
    )
    parser.add_argument("--head_hidden", type=int, default=256, help="BiLSTM hidden size per direction.")
    parser.add_argument("--head_layers", type=int, default=1, help="Number of BiLSTM layers.")
    parser.add_argument("--domain_dim", type=int, default=0, help="CL/IT embedding size; 0 turns it off.")
    parser.add_argument("--attn_heads", type=int, default=0, help="Sentence self-attention heads; 0 turns it off.")
    args = parser.parse_args()
    if args.include_dev and not args.fixed:
        raise SystemExit("--include_dev needs --fixed, because the dev score can no longer pick the epoch")
    if args.attn_heads and (2 * args.head_hidden) % args.attn_heads:
        raise SystemExit("2 * head_hidden must divide evenly by attn_heads")

    if args.refit_from and args.adapters_from:
        raise SystemExit("use only one of --refit_from and --adapters_from")
    refit = None
    adapters = None
    if args.refit_from:
        refit = torch.load(args.refit_from, map_location="cpu", weights_only=False)
        args.arch, args.rank, args.alpha = refit["arch"], refit["rank"], refit["alpha"]
        for key, value in head_options(refit).items():
            setattr(args, key, value)
        method = refit["method"]
        if not args.weights:
            args.weights = refit["weights"] if os.path.isfile(refit["weights"]) else DEFAULT_WEIGHTS[args.arch]
        print("refit from", args.refit_from, "dev", round(refit["dev_f1"], 4), flush=True)
    elif args.adapters_from:
        adapters = torch.load(args.adapters_from, map_location="cpu", weights_only=False)
        args.arch, args.rank, args.alpha = adapters["arch"], adapters["rank"], adapters["alpha"]
        method = adapters["method"]
        if not args.weights:
            args.weights = adapters["weights"] if os.path.isfile(adapters["weights"]) else DEFAULT_WEIGHTS[args.arch]
        print(
            "adapters from",
            args.adapters_from,
            "dev",
            round(adapters["dev_f1"], 4),
            "new head",
            args.head_hidden,
            args.head_layers,
            args.domain_dim,
            args.attn_heads,
            flush=True,
        )
    else:
        method = args.student_method if args.method == "distill" else args.method
    if args.method == "distill" and not args.teacher:
        raise SystemExit("distill needs --teacher pointing at a finished Phase 3 best.pt")
    cached_mode = method == "frozen" or refit is not None or adapters is not None
    args.weights = args.weights or DEFAULT_WEIGHTS[args.arch]
    args.lr = args.lr or DEFAULT_LR[method]
    if (refit is not None or adapters is not None) and args.head_lr == 1e-3:
        args.head_lr = 5e-4
    epochs = args.epochs or (40 if cached_mode else 8)
    patience = args.patience or (8 if cached_mode else 3)

    set_seed(args.seed)
    os.makedirs(args.out, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device", device, "arch", args.arch, "method", args.method, "adapter", method, "seed", args.seed, flush=True)
    print("weights", args.weights, "lr", args.lr, "head_lr", args.head_lr, flush=True)

    vocab = read_vocab(args.data_dir)
    label_index = {label: position for position, label in enumerate(vocab)}
    train_docs = load_docs(args.data_dir, TRAIN_FILES)
    dev_docs = load_docs(args.data_dir, DEV_FILES)
    test_docs = load_docs(args.data_dir, TEST_FILES)
    every_doc = train_docs + dev_docs + test_docs
    if args.include_dev:
        train_docs = train_docs + dev_docs
    print("docs train", len(train_docs), "dev", len(dev_docs), "test", len(test_docs), flush=True)

    tokenizer = LegalBPE.load(args.bpe)
    tokenize_docs(tokenizer, every_doc, args.eval_tokens)

    teacher = None
    if args.teacher:
        teacher = teacher_scores(args, train_docs, len(vocab), device)

    tagger = build_tagger(
        args.arch,
        method,
        args.weights,
        args.rank,
        args.alpha,
        len(vocab),
        args.head_hidden,
        args.head_layers,
        args.domain_dim,
        args.attn_heads,
    ).to(device)
    if refit is not None:
        load_trained(tagger, refit["model"])
        for parameter in tagger.backbone.parameters():
            parameter.requires_grad_(False)
    elif adapters is not None:
        adapter_state = {
            name: value
            for name, value in adapters["model"].items()
            if name in tagger.adapter_keys or name.endswith(".mask")
        }
        missing = tagger.adapter_keys - set(adapter_state)
        if missing:
            raise SystemExit(f"adapters_from is missing keys: {list(missing)[:5]}")
        tagger.load_state_dict(adapter_state, strict=False)
        for parameter in tagger.backbone.parameters():
            parameter.requires_grad_(False)
    weights = class_weights(train_docs, vocab, args.weight_cap).to(device)
    print("class weights", {label: round(float(value), 2) for label, value in zip(vocab, weights)}, flush=True)

    head_params = [p for name, p in tagger.named_parameters() if not name.startswith("backbone.")]
    adapter_params = [p for name, p in tagger.named_parameters() if name.startswith("backbone.") and p.requires_grad]
    groups = [{"params": head_params, "lr": args.head_lr}]
    if adapter_params:
        groups.append({"params": adapter_params, "lr": args.lr})
    backbone_total = sum(p.numel() for p in tagger.backbone.parameters())
    print(
        f"Phase 1 parameters {backbone_total / 1e6:.2f}M  trainable adapter {sum(p.numel() for p in adapter_params) / 1e6:.3f}M  "
        f"head {sum(p.numel() for p in head_params) / 1e6:.2f}M",
        flush=True,
    )
    optimizer = torch.optim.AdamW(groups, weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    best_path = os.path.join(args.out, "best.pt")
    all_params = [p for group in groups for p in group["params"]]

    cached = None
    if cached_mode:
        started = time.time()
        cached = doc_vectors(tagger, every_doc, device)
        print(f"cached sentence vectors {time.time() - started:.0f}s", flush=True)

    best_f1 = -1.0
    if refit is not None or adapters is not None:
        dev_pred = predict(tagger, dev_docs, cached, vocab, device)
        best_f1 = macro_f1(dev_docs, dev_pred, vocab)[0]
        print(f"epoch 0 dev macro-F1 {best_f1:.4f}  ({domain_scores(dev_docs, dev_pred, vocab)})", flush=True)
        if refit is not None:
            save_checkpoint(best_path, tagger, args, method, vocab, best_f1)
    waited = 0
    run_started = time.time()
    for epoch in range(1, epochs + 1):
        started = time.time()
        tagger.train()
        backbone_mode(tagger, "frozen" if cached_mode else method, True)
        total = 0.0
        steps = 0
        if cached_mode:
            pieces = [(doc, 0, len(doc["ids"])) for doc in train_docs]
            random.shuffle(pieces)
        else:
            pieces = training_chunks(train_docs, args.chunk)
        for doc, start, stop in pieces:
            tags = torch.tensor([label_index[label] for label in doc["labels"][start:stop]], device=device)
            positions = positions_of(len(doc["ids"]), start, stop - start).to(device)
            if cached_mode:
                vectors = cached[doc["id"]][start:stop].to(device)
            else:
                rows = [ids[: args.train_tokens] for ids in doc["ids"][start:stop]]
                token_ids, pad_mask = pad_batch(rows, device)
                with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                    vectors = tagger.encode(token_ids, pad_mask)
            emissions = tagger.emissions(vectors.float(), positions, doc["domain"])
            loss = role_loss(tagger, emissions, tags, weights, args.ce_weight)
            if teacher is not None:
                soft = teacher[doc["id"]][start:stop].to(device)
                loss = (1 - args.kd_weight) * loss + args.kd_weight * distill_loss(emissions, soft, args.kd_temp)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(all_params, 1.0)
            scaler.step(optimizer)
            scaler.update()
            total += loss.item()
            steps += 1
            if not cached_mode and steps % 100 == 0:
                print(f"  step {steps}/{len(pieces)} loss {total / steps:.4f} {time.time() - started:.0f}s", flush=True)
            if args.max_steps and steps >= args.max_steps:
                break
        print(f"epoch {epoch} train loss {total / max(1, steps):.4f} {time.time() - started:.0f}s", flush=True)

        vectors = cached if cached_mode else doc_vectors(tagger, dev_docs, device)
        dev_pred = predict(tagger, dev_docs, vectors, vocab, device)
        f1, lines = macro_f1(dev_docs, dev_pred, vocab)
        print(f"epoch {epoch} dev macro-F1 {f1:.4f}  ({domain_scores(dev_docs, dev_pred, vocab)})", flush=True)
        if args.fixed:
            if epoch == epochs:
                save_checkpoint(best_path, tagger, args, method, vocab, f1)
        elif f1 > best_f1:
            best_f1 = f1
            waited = 0
            save_checkpoint(best_path, tagger, args, method, vocab, f1)
        else:
            waited += 1
            if waited >= patience:
                print("early stop", flush=True)
                break
    print(f"training time {(time.time() - run_started) / 60:.1f} min", flush=True)

    checkpoint = torch.load(best_path, map_location="cpu", weights_only=False)
    load_trained(tagger, checkpoint["model"])
    tagger.to(device)
    if not cached_mode:
        cached = doc_vectors(tagger, dev_docs + test_docs, device)
    dev_pred = predict(tagger, dev_docs, cached, vocab, device)
    f1, lines = macro_f1(dev_docs, dev_pred, vocab)
    print(f"best checkpoint dev macro-F1 {f1:.4f}  ({domain_scores(dev_docs, dev_pred, vocab)})", flush=True)
    print("\n".join(lines), flush=True)
    write_dev_files(args.out, dev_docs, dev_pred)
    test_pred = predict(tagger, test_docs, cached, vocab, device)
    for doc in test_docs:
        if len(test_pred[doc["id"]]) != len(doc["text"]):
            raise SystemExit(f"{doc['id']}: {len(test_pred[doc['id']])} roles for {len(doc['text'])} sentences")
    write_outputs(args.out, test_pred)


In [ ]:
_register('phase3_rr_own')


## 4. Serving path


In [ ]:
print("efficient + LoRMA + BiLSTM-CRF + rarity bias")
print("prediction.json: {doc_id: [role per official sentence]}")
